# Court Complaint Information Extraction with hybrid RAG

*Stage 2 of the court-complaint extraction project. Stage 1 (`01_Separate_Complaints.ipynb`) selects one complaint per case from a downloaded library; this notebook extracts structured information from each complaint, with every value checked against a quote from the document.*

## Context

Litigation research often requires coding large numbers of court complaints by hand: reading each filing to record the case details and the parties' characteristics. This notebook adapts a hybrid RAG pipeline to extract structured, citation-checked information from a court complaint PDF. It is designed as a prototype for processing a large library of complaints, such as those downloaded from commercial legal databases.

## Objective

For each complaint, produce a DataFrame with one row per plaintiff and these fields:

1. Case number
2. Case name
3. Date filed
4. Plaintiff name
5. Plaintiff age
6. Plaintiff gender
7. Plaintiff race
8. Defendant name
9. Encounter setting: where the profiling took place (primary setting, setting detail, all settings), whether it involved a traffic stop, and, for commercial settings, whether the plaintiff was treated as a criminal suspect

Every extracted value must be **grounded in the complaint** and pass an automated **citation check** before it is accepted.

## Handling of sensitive attributes

Age, gender and race are recorded **only when the complaint states them**. Race is never inferred from a name, location or any other cue. Gender is marked as *inferred* when it rests only on pronouns or honorifics, and it is never inferred from a name. Values the complaint does not state are recorded as "Not stated".

---
### How to run this notebook

1. **Runtime → Change runtime type → T4 GPU** (the reranker is the slowest step on a CPU: about 10 minutes per complaint on a CPU).
2. Have `config.json` ready (`{"OPENAI_API_KEY": "..."}`; add a base URL only if you use an API gateway).
3. In **Settings** (*Input Document and Model Selection*), `RUN = "full"` (the default: all sorted complaints) or `RUN = "validation"` (the gold-set complaints, already processed). The cell just above the last one shows (and can switch) the folder before processing.
4. Run the install cell → **Runtime → Restart session** → click the next code cell → **Runtime → Run cell and below**.
   - The last cell processes the folder and writes `library_analysis.xlsx` there; with `RUN = "validation"`, that is what `03_Validation.ipynb` compares with the hand coding.
   - **Each complaint is saved as soon as it is finished** (in `_results/`), so the run is **resumable**: if the session ends, run the notebook again and it continues with the remaining complaints; finished complaints are never re-processed or overwritten.

The configuration validated on the gold set is fixed in Settings (model `gpt-4o`, 60 pages indexed); keep it for the full run, or validate again after changing it.

## Pipeline Design

| Stage | What it does | Carried over from the earlier RAG project |
|---|---|---|
| **Load and clean at ingestion** | Reads each PDF page; removes pleading-paper line numbers, repeated footers and boilerplate, and the "Page X of Y" part of court (ECF) headers, while keeping the case number and filing date in those headers | Excerpt cleaning, moved to ingestion; removing competing page numbers |
| **Token-based chunking per page** | 350-token chunks with 50-token overlap; each chunk keeps its PDF page number | Token-based chunking with correct page metadata |
| **Hybrid retrieval + reranking** | BM25 keyword search (names, case numbers) + MMR semantic search, fused and reranked by a cross-encoder | Hybrid search, MMR, reranking |
| **Field-specific multi-query retrieval** | Each group of fields is retrieved with several targeted queries; the caption pages are always included for case and party fields | Multi-query retrieval (adopted here because complaint wording varies widely) |
| **Structured extraction** | The LLM returns JSON: for each field, a value, the source page, a verbatim quote, and a basis (explicit / inferred / not stated) | Grounded system prompt with strict rules |
| **Citation check** | For every value: (1) the cited page was among the excerpts, (2) the quote appears verbatim on that page, (3) the quote supports the value | Citation validation, extended from page-level to quote- and value-level |
| **Automatic correction** | Values that fail the check are sent back to the model once, with the reasons | Correction step |
| **Output** | `complaint_df`: one row per plaintiff, with verified values only; `audit_df`: every value with its page, quote and check results | Pipeline summary |

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Remove pre-installed packages this notebook doesn't use (avoids dependency-conflict warnings)
!pip uninstall -y -q langgraph langgraph-prebuilt langgraph-sdk google-adk

# Install required libraries
# openai is pinned below 2.0 because langchain_openai 0.3.30 requires openai >=1.99.9,<2.0.0
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              langchain_openai==0.3.30 \
              "openai>=1.99.9,<2.0.0" \
              rank_bm25==0.2.2

**Note**:
- After running the above cell, restart the runtime (Runtime → Restart session) and run all cells sequentially from the next cell.
- Warnings about dependency conflicts with other pre-installed Colab packages (e.g., google-adk, langgraph, gcsfs) can be ignored.

In [ ]:
# Verify library versions after restarting the runtime (openai must start with 1.)
import openai
from importlib.metadata import version
print("openai:", openai.__version__)
print("langchain_openai:", version("langchain-openai"))

In [ ]:
# Core libraries
import os, re, json, math, uuid, glob, difflib
from collections import Counter
import numpy as np
import pandas as pd
from dateutil import parser as date_parser

# OpenAI
from openai import OpenAI

# LangChain: loading, chunking, embeddings, vector DB, retrievers
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

## Setting up the OpenAI API

In [ ]:
CONFIG_PATH = '/content/config.json'

# Upload only if the file isn't already in this session
if not os.path.exists(CONFIG_PATH):
    from google.colab import files
    print("config.json not found - please select it in the dialog below")
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    if name != 'config.json':
        os.rename(f'/content/{name}', CONFIG_PATH)

with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

os.environ['OPENAI_API_KEY'] = config["OPENAI_API_KEY"]
if config.get("OPENAI_API_BASE"):
    os.environ["OPENAI_BASE_URL"] = config["OPENAI_API_BASE"]

client = OpenAI()
print("Credentials loaded. Base URL set:", "OPENAI_BASE_URL" in os.environ)

In [ ]:
def parse_json(text):
    """Parse the model's JSON output; fall back to the first {...} block if extra text surrounds it."""
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        match = re.search(r"\{.*\}", text or "", flags=re.S)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    return {}


def response_json(user_prompt, system_prompt, model="gpt-4o-mini", max_tokens=2500):
    """Call the chat model in JSON mode (falls back to plain mode if the API endpoint does not support it)."""
    kwargs = dict(
        model=model,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
        max_tokens=max_tokens,
        temperature=0,
    )
    try:
        completion = client.chat.completions.create(response_format={"type": "json_object"}, **kwargs)
    except Exception as e:
        if "response_format" not in str(e):
            raise
        completion = client.chat.completions.create(**kwargs)
    return parse_json(completion.choices[0].message.content)

## Input Document and Model Selection

This cell chooses the complaints to process and the models used by the pipeline. The default is `RUN = "full"` (all sorted complaints in `complaints/`); `RUN = "validation"` processes the gold-set complaints in `validation_complaints/` (prepared by Step 6b of `01_Separate_Complaints.ipynb`); `RUN = "full"` processes all sorted complaints in `complaints/`. The single-complaint walkthrough below uses the first PDF in the folder.

## Chunking and Model Choices

**Chunking (350 tokens, 50-token overlap).** The fields this pipeline extracts are short facts, often stated in a single numbered paragraph (e.g., "Plaintiff Eric André is a Black ... comedian"). Chunks of about 350 tokens (roughly one to three paragraphs) are small enough that each one is about a single topic, which makes retrieval more precise. Each page is chunked separately, so every chunk keeps its PDF page number for citations. The 50-token overlap (about 15%) repeats a sentence or two at each boundary, so a fact that falls between two chunks still appears whole in at least one of them.

**Extraction model (`gpt-4o`).** The model is fixed to the one validated on the gold set, so the validation results apply to the full run. Stronger models handle long, multi-party complaints better; every value is citation-checked regardless of the model. To use another model (e.g. `gpt-4o-mini`, about 15 times cheaper), change `MODEL_PREFERENCE` and validate it on the gold set first.

**Embedding model (`text-embedding-3-small`).** OpenAI's current small embedding model offers strong retrieval quality at very low cost, which matters when processing hundreds of complaints. Domain-specific legal embedding models exist, including commercial options such as Kanon 2 Embedder and voyage-law-2 and open-source models on Hugging Face, and they lead legal retrieval benchmarks. Those benchmarks, however, test harder tasks, such as finding relevant case law across large collections. This pipeline searches one complaint at a time for plainly worded facts, and keyword search (BM25), the caption-page anchors, and the cross-encoder reranker already locate them reliably. A general-purpose OpenAI embedding model is therefore sufficient; a legal model would be worth adopting only if validation against hand-coded complaints showed it reduces missed fields.

**Reranker (`BAAI/bge-reranker-base`).** An open-source cross-encoder that runs locally at no API cost. It reads the query and each candidate chunk together to select the most relevant excerpts, improving precision after the broader hybrid search.

**Validation run:** `COMPLAINT_FOLDER` points to the 21 gold-set complaints. The walkthrough cells below process the first of them as an example; the batch cell at the end processes all of them and writes `library_analysis.xlsx`, which `03_Validation.ipynb` compares with the hand coding.

**`MAX_PAGES_INDEXED`** (default 60): for files longer than this, only the first pages are indexed. Complaints state the parties, the plaintiff's description and the encounter at the start, and long files usually continue with exhibits or other filings, so this mainly cuts embedding and indexing time for those files. Each complaint's output records how many pages were indexed (`Pages indexed`).

In [ ]:
import sys
import urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                        # complaints and results on Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

# ------------------------------------------------------------
# Complaint PDF
# ------------------------------------------------------------
RUN = "full"                # "full": all sorted complaints | "validation": the gold-set complaints (done)
FOLDERS = {"validation": "/content/drive/MyDrive/COMPLAINTS_SORTED/validation_complaints",
           "full": "/content/drive/MyDrive/COMPLAINTS_SORTED/complaints"}
COMPLAINT_FOLDER = FOLDERS[RUN]
assert os.path.isdir(COMPLAINT_FOLDER), (f"Folder not found: {COMPLAINT_FOLDER}"
                                         " (for the validation run, run Step 6b of 01_Separate_Complaints.ipynb first)")
PDFS = sorted(glob.glob(os.path.join(COMPLAINT_FOLDER, "*.pdf")))
print(f"RUN = {RUN}: {len(PDFS)} PDFs in {COMPLAINT_FOLDER}")

# The single-complaint walkthrough below uses the first PDF in the folder (nothing is downloaded into it)
PDF_PATH = PDFS[0]

# ------------------------------------------------------------
# Extraction model: the validated model (gpt-4o)
# ------------------------------------------------------------
MODEL_PREFERENCE = ["gpt-4o"]      # the model validated on the gold set; change only with a new validation

def first_allowed_model(candidates):
    """Return the first model the API key can use (tested with a tiny request)."""
    for m in candidates:
        try:
            client.chat.completions.create(
                model=m, messages=[{"role": "user", "content": "Say OK"}], max_tokens=3
            )
            return m
        except Exception as e:
            print(f"  {m}: not available ({str(e)[:70]})")
    raise RuntimeError("None of the candidate models is available to this API key.")

EXTRACTION_MODEL = first_allowed_model(MODEL_PREFERENCE)

# ------------------------------------------------------------
# Other settings
# ------------------------------------------------------------
EMBEDDING_MODEL = "text-embedding-3-small"
RERANKER_MODEL = "BAAI/bge-reranker-base"
CHUNK_SIZE, CHUNK_OVERLAP = 350, 50                 # tokens
# Index only the first pages of very long files. The parties, the plaintiff's description and the encounter are
# stated at the start of a complaint; long files usually continue with exhibits or other filings. Most complaints
# are shorter than this (median about 30 pages), so they are indexed in full. None = index every page.
MAX_PAGES_INDEXED = 60

print("\nPDF found:", os.path.exists(PDF_PATH), "|", PDF_PATH)
print("Files in folder:", os.listdir(COMPLAINT_FOLDER))
print("Extraction model:", EXTRACTION_MODEL)

## Loading and Cleaning the Complaint

Court filings contain text added by the court's filing system or the document template, which is not part of the complaint. It creates competing page numbers, which can cause wrong citations, and repeated noise in every chunk. This cleaning step applies three rules to each page:

- **ECF headers** (e.g., `Case 1:22-cv-04065-MHC Document 24 Filed 12/30/22 Page 3 of 53`): the line is kept, because it contains the case number and filing date, but `Page 3 of 53` is removed, since the court's page count may not match the PDF pages used for citations.
- **Stand-alone numbers** (printed page numbers and pleading-paper line numbers): removed.
- **Repeated footers and boilerplate** (e.g., `COMPLAINT FOR DAMAGES - 3`): removed when a line, ignoring its numbers, appears on at least half the pages.

Lines with a case number or the word "Filed," and numbered paragraph markers, are always kept. Cleaning happens once, at ingestion, so retrieval, the model and the citation check all work from the same text.

In [ ]:
CASE_NUMBER_RE = re.compile(r"\b\d{1,2}:\d{2}-[a-z]{2,4}-\d{3,6}\b", re.I)    # federal format, e.g. 1:20-cv-01234
ECF_PAGE_RE = re.compile(r"\bPage\s*:?\s*\d+\s*of\s*\d+\b", re.I)          # 'Page 3 of 25'
STANDALONE_NUMBER_RE = re.compile(r"^\s*\d{1,3}\s*$")                        # pleading line numbers, page numbers


def _normalize_line(line):
    """Replace digits with '#' so repeated headers/footers match across pages."""
    return re.sub(r"\d+", "#", line.strip())


def _is_informative_header(line):
    """Header lines worth keeping: they carry the case number or the filing date."""
    return bool(CASE_NUMBER_RE.search(line)) or bool(re.search(r"\bFiled\b", line, re.I))


def detect_ocr_pages(pdf_path, min_cover=0.8):
    """Pages that are scanned images with a text layer (OCR): an image covers most of the page and text is present.
    OCR text contains recognition errors, so the citation check is more tolerant for these documents."""
    import fitz                                  # PyMuPDF
    ocr_pages = []
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc, start=1):
            page_area = page.rect.width * page.rect.height
            full_page_image = any(
                fitz.Rect(img["bbox"]).width * fitz.Rect(img["bbox"]).height >= min_cover * page_area
                for img in page.get_image_info())
            if full_page_image and page.get_text().strip():
                ocr_pages.append(i)
    return ocr_pages


def load_complaint(pdf_path):
    """Load a complaint PDF and clean each page.
    Returns: page_docs (one Document per page, metadata page = PDF page number, 1-based),
             page_texts {page: cleaned text}, and the set of repeated lines that were removed."""
    raw_pages = PyMuPDFLoader(pdf_path).load()
    n_pages = len(raw_pages)
    threshold = max(3, math.ceil(0.5 * n_pages))

    counts = Counter(
        norm for p in raw_pages
        for norm in {_normalize_line(l) for l in p.page_content.split("\n") if l.strip()}
    )
    repeated = {line for line, n in counts.items() if n >= threshold}

    page_docs, page_texts, removed = [], {}, set()
    for i, p in enumerate(raw_pages):
        kept = []
        for line in p.page_content.split("\n"):
            s = line.strip()
            if not s or STANDALONE_NUMBER_RE.fullmatch(s):
                continue
            if (_normalize_line(s) in repeated and re.search(r"[A-Za-z]{2,}", s)
                    and not _is_informative_header(s)):
                removed.add(s)
                continue
            s = re.sub(r"\s{2,}", " ", ECF_PAGE_RE.sub("", s)).strip()
            if s:
                kept.append(s)
        text = "\n".join(kept)
        page_no = i + 1
        page_texts[page_no] = text
        page_docs.append(Document(page_content=text,
                                  metadata={"page": page_no, "source": os.path.basename(pdf_path)}))
    return page_docs, page_texts, removed

In [ ]:
page_docs, page_texts, removed_lines = load_complaint(PDF_PATH)

print("Pages:", len(page_docs))
print("Repeated lines removed:", len(removed_lines))
for line in sorted(removed_lines)[:15]:
    print("   ", line[:100])
print("\n--- Page 1 (cleaned) ---\n" + page_texts[1][:1500])

## Chunking, Embedding and Retrieval

Each page is split into token-based chunks that keep their page number. A fresh in-memory vector store is built for each complaint and deleted after extraction, so complaints never mix when many files are processed.

**Why a recursive text splitter.** Complaints consist of numbered paragraphs, each usually stating one allegation or describing one party. The recursive splitter breaks text at the largest natural boundary first (paragraphs, then lines, then spaces), so a party's description usually stays whole within one chunk. Chunk size is measured with the embedding model's tokenizer, so the 350-token limit is exact.

**Why hybrid retrieval.** Complaints contain two kinds of information, and each needs a different search method:

- **Exact identifiers:** party names, case numbers, and fixed legal phrases such as "official capacity." Semantic embeddings capture meaning but can blur rare names and numbers; **BM25 keyword search** matches them exactly.
- **Facts described in varied wording:** the same attribute appears as "African American," "Black," or "a Black man"; age as "34-year-old" or "aged thirty-four." **Semantic search** finds these paraphrases even when they share no words with the query.

Combining the two means a passage is found whether the query and the complaint share exact words or only meaning.

**Why MMR.** Complaints are highly repetitive: plaintiffs' and defendants' names recur in dozens of paragraphs, mostly in the factual allegations. Plain similarity search would return many near-identical passages about the same events and crowd out the few that matter, such as the parties section. **MMR (maximal marginal relevance)** balances relevance against variety, so retrieved passages come from different parts of the complaint: the caption, the parties section, and the relevant allegations.

**Reranking.** A cross-encoder then reads each query together with each candidate passage and keeps the most relevant ones, adding precision after the broader search.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=config["OPENAI_API_KEY"],
    openai_api_base=config.get("OPENAI_API_BASE"),
)

_reranker_model = None
def get_reranker_model():
    """Load the cross-encoder once and reuse it for every complaint."""
    global _reranker_model
    if _reranker_model is None:
        import torch
        _reranker_model = HuggingFaceCrossEncoder(
            model_name=RERANKER_MODEL,
            model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
        )
    return _reranker_model


def _tokenize(text):
    return re.findall(r"\w+", text.lower())


def build_index(page_docs):
    """Chunk the complaint, embed it into a fresh vector store, and build the hybrid retriever."""
    chunks = text_splitter.split_documents(page_docs)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
    n = len(chunks)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name=f"complaint_{uuid.uuid4().hex[:12]}",   # unique per complaint
        collection_metadata={"hnsw:space": "cosine"},
    )
    dense = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": min(12, n), "fetch_k": min(30, n), "lambda_mult": 0.6},
    )
    bm25 = BM25Retriever.from_documents(chunks, preprocess_func=_tokenize)
    bm25.k = min(12, n)
    hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.4, 0.6])
    retriever = ContextualCompressionRetriever(
        base_compressor=CrossEncoderReranker(model=get_reranker_model(), top_n=8),
        base_retriever=hybrid,
    )
    return chunks, vectorstore, retriever

## Extraction Fields and Retrieval Queries

Fields are extracted in three groups, because the information sits in different parts of a complaint:

- **Case-level fields** (case number, case name, filing date, document title) appear on the caption page and in the court headers.
- **Party fields** (each plaintiff's name, age, gender and race, and each defendant's name) are spread across the caption, the parties section and the factual allegations.
- **Setting fields** (where the profiling took place) are in the factual allegations; the caption is not used for them.

Each group has its own retrieval queries and its own prompt. This keeps the long factual allegations from crowding out the short parties section.

**Multiple queries per group.** Complaints describe the same facts in different words (e.g., "African American" or "Black"; "a 34-year-old man" or "aged thirty-four"), so each group is searched with several phrasings.

**Caption pages always included.** The caption is the one part of every complaint with a fixed location and format, listing the case number, case name and all parties, so it is added to every search.

In [ ]:
# Defining field groups

FIELD_GROUPS = {
    "case": {
        "anchor_pages": [1, 2],
        "top_k": 8,
        "queries": [
            "case number civil action no. docket number",
            "caption plaintiff versus defendant complaint for damages",
            "date filed filing stamp document filed",
            "complaint title first amended complaint jury demand",
        ],
        "instructions": """Extract the case-level fields:
- case_number: the docket / civil action number exactly as printed.
- case_name: the caption as 'First Plaintiff v. First Defendant', adding 'et al.' if more parties are named on that side.
- date_filed: the date THIS document was filed (value = the date only, e.g. "12/08/23", without words such as "Filed"), preferably from a court filing stamp, docket cover, or ECF header (e.g. 'Filed 03/04/20'). Use a signature date ('Dated: ...') only if no filing date appears, and then set basis to "inferred".
- document_title: the document's title as printed (e.g. "COMPLAINT", "FIRST AMENDED COMPLAINT"), so that amended complaints can be identified.

Return JSON exactly in this shape:
{"case_number": FIELD, "case_name": FIELD, "date_filed": FIELD, "document_title": FIELD}
where FIELD = {"value": ..., "page": P, "quote": "...", "basis": "explicit" | "inferred" | "not_stated"}""",
    },
    "parties": {
        "anchor_pages": [1],
        "top_k": 12,
        "queries": [
            "parties plaintiff is an individual resident of",
            "plaintiff is a year-old man woman",
            "plaintiff race African American Black Hispanic Latino White Asian Native American",
            "plaintiff age years old",
            "Mr. Ms. Mrs. plaintiff he she his her",
            "defendant city county police department officer sued in individual and official capacity",
            "defendants are",
        ],
        "instructions": """Extract every plaintiff and every defendant named as a PARTY in the caption or the parties section.
Complaints often describe other people (other victims, witnesses, officials who are not sued); never list them as plaintiffs or defendants, and never take a plaintiff's age, gender, or race from a passage about someone else. Each plaintiff's age, gender, and race quote must refer to that plaintiff (by name, or as "Plaintiff"/"Plaintiffs").
For each plaintiff: name, age, gender, race.
For each defendant: name (individual officers with rank if given, agencies, municipalities, companies).
- age: only a specific age stated in the text (e.g. "34", "thirty-four"); statements such as "an adult" or "over the age of eighteen" are not an age, so return not_stated. Never calculate age from a date of birth or other dates.
- gender: "Male", "Female", or the complaint's own term. If no explicit term (man, woman, male, female) is used for a plaintiff, quote a passage with an honorific and the plaintiff's surname (e.g. "Ms. Smith", "Mr. Jones"), which identifies both the gender and the person. If the complaint uses only pronouns, quote ONE sentence in which the pronoun appears together with "Plaintiff" or the plaintiff's name (e.g. "Plaintiff told the officers that she owned the gun"), and set basis to "inferred".
- race: the complaint's own term (e.g. "African American", "Black", "Latino").

Return JSON exactly in this shape:
{"plaintiffs": [{"name": FIELD, "age": FIELD, "gender": FIELD, "race": FIELD}],
 "defendants": [{"name": FIELD}]}
where FIELD = {"value": ..., "page": P, "quote": "...", "basis": "explicit" | "inferred" | "not_stated"}""",
    },
    "setting": {
        "anchor_pages": [],
        "top_k": 12,
        "queries": [
            "plaintiff was pulled over traffic stop vehicle driving car",
            "officers approached stopped detained plaintiff while walking street",
            "airport flight boarding TSA jet bridge border checkpoint",
            "store shopping customer security accused shoplifting restaurant hotel bank",
            "plaintiff home residence apartment house police entered",
            "school campus workplace government building",
            "policy practice pattern program of stopping",
            "police station jail custody booking holding cell strip search",
            "home house residence raided search warrant entered",
            "employment employer position promotion discipline Title VII discrimination",
            "facts on or about the date plaintiff was",
        ],
        "instructions": """First decide the scope of the complaint, then classify where the alleged racial profiling took place, using this codebook.

Scope (case_scope):
- encounter: the complaint describes one or more encounters in which a plaintiff was stopped, approached, questioned, searched, detained, arrested, followed, or refused service
- employment: the complaint is about the plaintiff's employment (e.g. an officer or employee suing their employer for discrimination)
- other: any other kind of complaint (e.g. a challenge to a policy only)
If case_scope is not "encounter", return not_stated for all setting fields below.

Level-1 setting codes:
- T: vehicle / transportation (the plaintiff was in or with a vehicle: driving, a passenger, or parked)
- P: pedestrian / public space (on foot in a public place, including public transit)
- J: travel hub or border (airport, rail or bus terminal, port of entry, immigration enforcement)
- C: commercial / public accommodation (store, restaurant, hotel, bank, other business open to the public)
- R: residential (the plaintiff's home, a housing complex or neighborhood, access to housing)
- I: institutional (school or campus, workplace, government or public facility)
- S: systemic (a challenge to a policy or program with no individual encounter described)

Level-2 sub-codes:
T1 traffic stop, T2 highway or drug interdiction, T3 checkpoint, T4 parked or stationary vehicle;
P1 street stop / stop-and-frisk, P2 park or public recreation, P3 public transit;
J1 airport or air travel, J2 border or immigration enforcement, J3 rail or bus terminal;
C1 retail store, C2 restaurant, bar or hotel, C3 bank or financial institution, C4 other business or service;
R1 own home or property, R2 housing complex or neighborhood, R3 housing access;
I1 school or university, I2 workplace, I3 government or public facility, I4 police station, jail or custody.

Fields:
- case_scope: "encounter", "employment" or "other", with a quote showing it.
- primary_setting: the Level-1 code of the place where the profiling contact BEGAN. The quote must NAME the place or the kind of encounter (e.g. "Officer Smith pulled Plaintiff over", "officers entered her home", "on the sidewalk outside the store"), not only describe the interaction. Never infer the setting from who the defendants are (a highway patrol defendant does not establish a traffic stop). If the excerpts do not establish the setting, return not_stated.
  Outside a business: code P if the encounter took place outside the premises (e.g. on the sidewalk or in the parking lot in front of a restaurant) and did not involve the plaintiff as a customer or the business's staff; code C only if the plaintiff was a customer or the business initiated the encounter.
- primary_setting_detail: the Level-2 sub-code for the primary setting, or not_stated if the sub-type is unclear.
- settings: one entry for EACH distinct Level-1 setting in which a plaintiff's encounter took place (including the primary one), each with its own quote. Include places the plaintiff was taken afterwards, such as a police station or jail (I, sub-type I4).
- traffic_stop: "yes" if any plaintiff's encounter involved a vehicle stop (quote the passage describing the stop); "no" only if the encounters described clearly did not involve a vehicle (then page and quote are null and basis is "inferred"); otherwise not_stated.
- pattern_alleged: "yes" if the complaint alleges that the conduct reflects a policy, practice, custom, program or pattern (e.g. describes other people treated the same way, statistics, or an enforcement program targeting a group), with a quote; otherwise not_stated.
- criminal_suspicion: only for commercial settings (C). "present" if the plaintiff was accused of or treated as suspected of a crime (e.g. shoplifting, followed, searched, detained, police or security called), with a quote; "absent" if the complaint describes only refused or poor service (page and quote null, basis "inferred"); otherwise not_stated.

Return JSON exactly in this shape:
{"case_scope": FIELD, "primary_setting": FIELD, "primary_setting_detail": FIELD, "settings": [FIELD, ...],
 "traffic_stop": FIELD, "pattern_alleged": FIELD, "criminal_suspicion": FIELD}
where FIELD = {"value": ..., "page": P, "quote": "...", "basis": "explicit" | "inferred" | "not_stated"}""",
    },
}


### Encounter Setting (codebook v0.3, lean version)

The setting fields classify **where the alleged profiling took place**, using a two-level codebook developed for this project. No published codebook classifies racial profiling incidents across settings; existing studies each cover one domain (police cases: Gabbidon, Marzette & Peterson, 2007; retail cases: Harris, Henderson & Williams, 2005).

| Level 1 | Setting | Level 2 sub-codes |
|---|---|---|
| **T** | Vehicle / transportation | T1 traffic stop · T2 highway or drug interdiction · T3 checkpoint · T4 parked or stationary vehicle |
| **P** | Pedestrian / public space | P1 street stop / stop-and-frisk · P2 park or public recreation · P3 public transit |
| **J** | Travel hub or border | J1 airport or air travel · J2 border or immigration enforcement · J3 rail or bus terminal |
| **C** | Commercial / public accommodation | C1 retail store · C2 restaurant, bar or hotel · C3 bank · C4 other business or service |
| **R** | Residential | R1 own home or property · R2 housing complex or neighborhood · R3 housing access |
| **I** | Institutional | I1 school or university · I2 workplace · I3 government or public facility · I4 police station, jail or custody |
| **S** | Systemic (no single encounter) | — |

Seven variables are coded per complaint:

| Variable | Values |
|---|---|
| Case scope | profiling encounter / employment / other; setting variables apply only to encounters |
| Primary setting | Level-1 code of the place where the profiling contact began |
| Setting detail | Level-2 sub-code, or not stated if the sub-type is unclear |
| All settings | every Level-1 setting in which a plaintiff's encounter took place |
| Traffic stop | yes / no / not stated |
| Pattern alleged | yes / not stated: the complaint alleges a policy, practice, custom, program or pattern (e.g., other people treated the same way, an enforcement program), the distinction between individual conduct and agency practice |
| Criminal suspicion (C only) | present / absent / not stated (from Harris, Henderson & Williams, 2005) |

**Changes in v0.2** (after the first pilot run on three complaints): a case-scope variable, because a library of racial profiling complaints also contains employment discrimination cases brought by officers against their departments; a custody sub-code (I4), recorded as a secondary setting when a plaintiff is taken to a police station or jail; and broader cue words (e.g., "home" without a possessive), after a correct residential code was rejected.

**Changes in v0.3** (after hand-checking four complaints): a pattern-or-practice flag, after a complaint alleged an airport interdiction program targeting Black passengers; and a rule for encounters outside a business, after a hand-checked encounter that took place outside a business (not involving it) could have been miscoded as commercial.

**Outside a business.** Code **P** if the encounter took place outside the premises and did not involve the plaintiff as a customer or the business's staff; code **C** only if the plaintiff was a customer or the business initiated the encounter.

**Coding rules.** The setting must be described in the quote, by naming the place or the kind of encounter; it is never inferred from the defendants' identity. If the complaint does not establish the setting, it is coded as not stated rather than guessed. "Traffic stop = no" and "criminal suspicion = absent" are accepted only when consistent with the verified settings, and are labeled *inferred*. Reliability should be reported at both levels: in prior content analysis, collapsed categories were coded more reliably than detailed ones (Moody-Hall, 2006).

In [ ]:
# Retrieving and labeling excerpts

def retrieve_for_group(retriever, chunks, group, per_query=4):
    """Multi-query retrieval: caption-page anchors first, then the top chunks for each query, de-duplicated."""
    seen, docs = set(), []

    def add(d):
        if d.metadata["chunk_id"] not in seen:
            seen.add(d.metadata["chunk_id"])
            docs.append(d)

    for page in group["anchor_pages"]:
        for d in [c for c in chunks if c.metadata["page"] == page][:2]:
            add(d)
    for query in group["queries"]:
        for d in retriever.invoke(query)[:per_query]:
            add(d)

    docs = docs[:group["top_k"]]
    return sorted(docs, key=lambda d: (d.metadata["page"], d.metadata["chunk_id"]))


def format_excerpts(docs):
    """Label each excerpt with its PDF page only, so the page is the only number available to cite."""
    return "\n\n".join(f"[Source page {d.metadata['page']}]\n{d.page_content}" for d in docs)

## Extraction Prompt

The system prompt sets the rules shared by both field groups. The model must use only the excerpts, support each value with a verbatim quote and the page from its label, and mark absent fields as "not stated" rather than guess. The output must be JSON, so every value can be parsed and citation-checked.

> **Sensitive attributes:** race, gender, and age are recorded only when the complaint states them, and are never inferred from names or other cues.

In [ ]:
EXTRACTION_SYSTEM = """You extract information from a court complaint for a legal research dataset, using ONLY the excerpts in the user message. Each excerpt starts with a label of the form [Source page P].

For every field return an object: {"value": ..., "page": P, "quote": "...", "basis": "explicit" | "inferred" | "not_stated"}.

Rules:
1. Use only the excerpts. Never use outside knowledge and never guess.
2. "quote" must be copied character-for-character from ONE excerpt (at most 40 words) and must contain the evidence for the value. "page" must be the P value from that excerpt's label. Never use any other number as a page, and ignore page numbers printed in the text.
3. If a field is not stated in the excerpts, return {"value": null, "page": null, "quote": null, "basis": "not_stated"}.
4. Copy names, case numbers and dates exactly as written.
5. Race or ethnicity: report only if the complaint explicitly states it, using the complaint's own term. Never infer race from a name, location, photograph, or any other cue.
6. Gender: prefer a quote with an explicit term (man, woman, male, female). If none exists, use an honorific with the plaintiff's surname (e.g., "Ms. Smith"); use a pronoun (he/she) only as a last resort, in a sentence that also names the plaintiff or says "Plaintiff". Never infer gender from a name.
7. Age: report only an age stated in the text. Do not calculate it from a date of birth or other dates.
8. Return valid JSON only, with no commentary."""

## Citation Check

Every extracted value goes through these checks before it is accepted:

1. **Page check:** the cited page is one of the pages whose excerpts the model received.
2. **Quote check:** the quote appears on that page. The comparison ignores case, spacing, punctuation and line-break hyphenation, so a quote copied across a line break still matches; a near-verbatim match (≥ 90%) is accepted and labeled as approximate. **Scanned (OCR) complaints**, detected automatically, contain recognition errors (e.g., "pennanent" for "permanent"), so for them the threshold is 80%; and for names, the value is also accepted if the name itself appears on the cited page.
3. **Support check:** the quote actually contains the value: the case number, every word of a name, the stated age (in digits or words, e.g. "thirty-seven"), the race term as written, a filing date matching the value, or a gender term (for inferred gender, a pronoun or honorific).
4. **Subject check** (plaintiff age, gender and race): the quote must refer to that plaintiff, by name or as "Plaintiff(s)," and the evidence term must appear close to that reference: within five words for gender and race, and ten words for age, which complaints phrase at greater distance (e.g., 'Plaintiff John Doe ("Doe") is an adult…'). Complaints often describe other people, such as officers or earlier victims of the same practice, and this check prevents their attributes from being assigned to a plaintiff. For example, "A female officer (Doe 3) first asked if Plaintiff owns a gun" is rejected as evidence of the plaintiff's gender, because "female" describes the officer.

**Setting fields** are categories rather than text, so their support check uses cue words: the quote must contain wording typical of the chosen code (e.g., "pulled over" or "traffic stop" for T1, "airport" or "boarding" for J1). In addition, a sub-code must belong to the verified primary setting, "traffic stop = no" is accepted only if none of the verified settings is a vehicle setting, and criminal suspicion is coded only when the verified primary setting is commercial. A "yes" for pattern alleged needs a quote alleging a policy, practice, program or pattern; its absence cannot be verified from text, so it is otherwise recorded as not stated. Setting fields are coded only when the verified case scope is a profiling encounter; for employment and other complaints they are marked not applicable.

A value is **verified** only if all checks pass. Values marked "not stated" need no citation. The check confirms that each value is supported by the text the model cites; it cannot detect a value the model wrongly reported as "not stated", which is why the audit table should be reviewed.

In [ ]:
def _compact(text):
    """Lowercase and keep only letters and digits: robust to spacing, punctuation and hyphenation."""
    return re.sub(r"[\W_]", "", str(text).lower())


def quote_on_page(quote, page_text, min_ratio=0.9):
    """Return 'exact', 'approximate', or None."""
    q, p = _compact(quote), _compact(page_text)
    if len(q) < 3:
        return None
    if q in p:
        return "exact"
    match = difflib.SequenceMatcher(None, p, q, autojunk=False).find_longest_match(0, len(p), 0, len(q))
    return "approximate" if match.size / len(q) >= min_ratio else None


_MONTHS = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?"
DATE_PATTERNS = [
    r"\b\d{1,2}/\d{1,2}/\d{2,4}\b",
    r"\b\d{4}-\d{2}-\d{2}\b",
    rf"\b{_MONTHS}\s+\d{{1,2}},?\s+\d{{4}}\b",
    rf"\b\d{{1,2}}\s+{_MONTHS}\s+\d{{4}}\b",
]

def parse_date(text):
    """Parse the first date found in the text (e.g. 'Filed 12/08/23' -> 2023-12-08); None if there is none."""
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return None
    s = str(text)
    for pattern in DATE_PATTERNS:
        m = re.search(pattern, s, flags=re.I)
        if m:
            try:
                return date_parser.parse(m.group(0)).date()
            except (ValueError, OverflowError):
                continue
    return None

def _date_in_text(value, text):
    target = parse_date(value)
    if target is None:
        return _compact(value) in _compact(text)
    return any(parse_date(m) == target
               for pattern in DATE_PATTERNS for m in re.findall(pattern, text, flags=re.I))


_UNITS = {w: i for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
    "fifteen sixteen seventeen eighteen nineteen".split())}
_TENS = {w: 10 * i for i, w in enumerate("_ _ twenty thirty forty fifty sixty seventy eighty ninety".split()) if i > 1}

def number_words_in(text):
    """Numbers written as words, e.g. 'thirty-seven-year-old' -> {37}."""
    found, words = set(), re.findall(r"[a-z]+", text.lower())
    for i, w in enumerate(words):
        if w in _TENS:
            nxt = words[i + 1] if i + 1 < len(words) else ""
            found.add(_TENS[w] + (_UNITS[nxt] if nxt in _UNITS and 0 < _UNITS[nxt] < 10 else 0))
        elif w in _UNITS and not (i > 0 and words[i - 1] in _TENS):
            found.add(_UNITS[w])
    return found


GENDER_TERMS = {
    "male":   {"male", "man", "men", "boy", "gentleman", "mr", "he", "him", "his", "himself",
               "father", "son", "husband", "brother"},
    "female": {"female", "woman", "women", "girl", "lady", "ms", "mrs", "miss", "she", "her", "hers",
               "herself", "mother", "daughter", "wife", "sister"},
}
EXPLICIT_GENDER_TERMS = {"male", "man", "men", "boy", "boys", "gentleman",
                         "female", "woman", "women", "girl", "girls", "lady"}
NAME_STOPWORDS = {"v", "vs", "et", "al"}

def value_supported(field_type, value, quote):
    """Does the quote contain the evidence for this value?"""
    q = quote or ""
    if field_type in ("setting_l1", "setting_l2"):
        pattern = SETTING_CUES.get(str(value))
        return bool(pattern) and bool(re.search(pattern, q, flags=re.I))
    if field_type == "pattern_alleged":
        return str(value).lower() == "yes" and bool(re.search(PATTERN_CUES, q, flags=re.I))
    if field_type == "case_scope":
        pattern = SCOPE_CUES.get(str(value).lower())
        return bool(pattern) and bool(re.search(pattern, q, flags=re.I))
    if field_type == "traffic_stop":
        return str(value).lower() == "yes" and bool(re.search(TRAFFIC_STOP_CUES, q, flags=re.I))
    if field_type == "criminal_suspicion":
        return str(value).lower() == "present" and bool(re.search(CRIMINAL_SUSPICION_CUES, q, flags=re.I))
    if field_type == "case_number":
        return _compact(value) in _compact(q)
    if field_type == "date_filed":
        return _date_in_text(value, q)
    if field_type == "age":
        numbers = re.findall(r"\d+", str(value))
        if not numbers:
            return _compact(value) in _compact(q)
        quote_numbers = set(re.findall(r"\d+", q)) | {str(n) for n in number_words_in(q)}
        return all(n in quote_numbers for n in numbers)
    if field_type == "gender":
        terms = GENDER_TERMS.get(str(value).strip().lower())
        words = set(re.findall(r"\w+", q.lower()))
        return bool(terms & words) if terms else _compact(value) in _compact(q)
    # names, case name, race: every meaningful word of the value must appear in the quote
    tokens = [t for t in re.findall(r"\w+", str(value)) if t.lower() not in NAME_STOPWORDS]
    return bool(tokens) and all(t.lower() in q.lower() for t in tokens)


# Maximum distance (in words) between the evidence term and the plaintiff reference.
# Age is phrased at greater distance ('Plaintiff X ("X") is an adult aged 34'), so its window is wider.
SUBJECT_WINDOWS = {"gender": 5, "race": 5, "age": 10}

def mentions_subject(quote, subject_name, field_type=None, value=None, window=None):
    """The quote must refer to this plaintiff (by name or as 'Plaintiff(s)'), and for gender/race/age
    the evidence term must appear within `window` words of that reference, so attributes of other
    people mentioned in the same quote (e.g. 'a female officer') are not assigned to the plaintiff."""
    window = window or SUBJECT_WINDOWS.get(field_type, 5)
    words = re.findall(r"\w+", (quote or "").lower())
    name_parts = {t for t in re.findall(r"\w+", str(subject_name or "").lower()) if len(t) > 2}
    subject_pos = [i for i, w in enumerate(words) if w in name_parts or w in ("plaintiff", "plaintiffs")]
    if not subject_pos:
        return False
    if field_type == "gender":
        # only terms matching the reported value (e.g. Female -> female, woman, she, her, ms ...)
        terms = GENDER_TERMS.get(str(value).strip().lower()) or set(re.findall(r"\w+", str(value).lower()))
    elif field_type == "race":
        terms = {t for t in re.findall(r"\w+", str(value).lower()) if len(t) > 2}
    elif field_type == "age":
        terms = set(re.findall(r"\d+", str(value))) | {w for w in words if w in _UNITS or w in _TENS}
    else:
        return True
    evidence_pos = [i for i, w in enumerate(words) if w in terms]
    return any(abs(e - s) <= window for e in evidence_pos for s in subject_pos)


# ---- Encounter setting: codebook v0.1 (lean) ----
SETTING_LEVEL1 = {
    "T": "Vehicle / transportation", "P": "Pedestrian / public space", "J": "Travel hub or border",
    "C": "Commercial / public accommodation", "R": "Residential", "I": "Institutional",
    "S": "Systemic (no single encounter)",
}
SETTING_LEVEL2 = {
    "T1": "Traffic stop", "T2": "Highway or drug interdiction", "T3": "Checkpoint", "T4": "Parked or stationary vehicle",
    "P1": "Street stop / stop-and-frisk", "P2": "Park or public recreation", "P3": "Public transit",
    "J1": "Airport or air travel", "J2": "Border or immigration enforcement", "J3": "Rail or bus terminal",
    "C1": "Retail store", "C2": "Restaurant, bar or hotel", "C3": "Bank or financial institution",
    "C4": "Other business or service",
    "R1": "Own home or property", "R2": "Housing complex or neighborhood", "R3": "Housing access",
    "I1": "School or university", "I2": "Workplace", "I3": "Government or public facility",
    "I4": "Police station, jail or custody",
}
CASE_SCOPES = {"encounter": "Profiling encounter", "employment": "Employment", "other": "Other"}
# Cue words: the quote must contain at least one cue for the chosen code (support check)
SETTING_CUES = {
    "T1": r"pull(ed|ing) (him|her|them|plaintiffs?|the \w+ )?over|traffic stop|traffic violation|motor vehicle stop|"
          r"speeding|tail ?light|license plate|stopped (his|her|their|the|plaintiffs?'?s?) (car|vehicle|truck|van)|"
          r"initiated a stop|traffic infraction|traffic citation",
    "T2": r"interdiction|drug courier|courier|k-?9|canine|\binterstate\b|\bi-\d+\b",
    "T3": r"checkpoint|road ?block",
    "T4": r"\bparked\b|sitting in (his|her|their|the) (car|vehicle|truck)",
    "P1": r"stop[- ]and[- ]frisk|frisk|pat(ted)?[- ]down|\bwalking\b|on foot|terry stop|pedestrian|\bstreet\b|sidewalk|"
          r"\bcorner\b|outside (the|a|his|her)\b|in public|public (place|area|property)",
    "P2": r"\bpark\b|playground|\bpool\b|\bbeach\b|barbecue|festival",
    "P3": r"\bbus\b|subway|\bmetro\b|transit|light rail|train",
    "J1": r"airport|\btsa\b|flight|boarding|jet ?bridge|jetway|airline|\bplane\b|aircraft|\bgate\b",
    "J2": r"\bborder\b|\bcbp\b|customs|immigration|\bice\b|port of entry",
    "J3": r"amtrak|greyhound|train station|bus (terminal|station)|\bterminal\b",
    "C1": r"\bstore\b|\bshop|retail|merchandise|shoplift|receipt|loss prevention|\bmall\b|cashier|checkout",
    "C2": r"restaurant|\bbar\b|hotel|motel|diner|\bcaf[eé]\b|waiter|waitress|\bserver\b",
    "C3": r"\bbank\b|teller|cash(ing)? a check|\baccount\b|\batm\b|credit union",
    "C4": r"gas station|\bgym\b|\buber\b|\blyft\b|salon|car wash|pharmacy|business|customer",
    "R1": r"\bhomes?\b|\bhouse\b|residence|front porch|driveway|\byard\b|\bdoorstep\b|front door",
    "R2": r"apartment complex|\bhoa\b|homeowners'? association|\bneighbor|complex|neighborhood|security guard",
    "R3": r"rental|\blease\b|landlord|tenant|crime[- ]free|housing",
    "I1": r"school|campus|university|college|\bstudent|dormitory",
    "I2": r"\bat work\b|workplace|employer|\bjob\b|\boffice\b|employee",
    "I3": r"courthouse|city hall|hospital|library|government building|\bdmv\b|post office",
    "I4": r"police station|\bstation\b|precinct|\bjail\b|custody|holding cell|booking|detention (center|facility)|"
          r"\bcell\b|sally ?port|lock-?up|strip[- ]search",
    "S":  r"policy|practice|pattern|program|custom|database|surveillance|systematic",
}
for _l1 in "TPJCRI":
    SETTING_CUES[_l1] = "|".join(f"(?:{p})" for k, p in SETTING_CUES.items() if len(k) == 2 and k[0] == _l1)
SETTING_CUES["T"] += r"|\bvehicle\b|\bcar\b|\bdriv(ing|er|e)\b|\bpassenger\b|\btruck\b|\bhighway\b"
SETTING_CUES["P"] += r"|\bstreet\b|sidewalk"
TRAFFIC_STOP_CUES = "|".join(f"(?:{SETTING_CUES[k]})" for k in ("T1", "T2", "T3")) + \
    r"|stopped .{0,40}\b(car|vehicle|truck|van)\b|\b(car|vehicle)\b.{0,40}\bstopped\b"
SCOPE_CUES = {
    "encounter": r"stop(ped)?\b|pulled|approach|question|search|detain|arrest|frisk|follow|accus|raid|"
                 r"handcuff|refused (to serve|service)|ordered|confront|seiz|interaction|encounter|"
                 r"\bcontact\b|\bcited\b|citation|ticket",
    "employment": r"employ|hired|\bfired\b|terminat|promot|demot|disciplin|\bposition\b|\bjob\b|title vii|eeoc|"
                  r"\bsergeant\b|workplace|supervisor|salary|\brank\b",
    "other": r".",
}
PATTERN_CUES = (r"\bpolic(y|ies)\b|\bpractices?\b|\bcustoms?\b|\bpatterns?\b|\bprograms?\b|routinely|systematic|"
                r"widespread|similarly situated|other (black|african|minority|hispanic|latino|people|motorists|"
                r"drivers|passengers|travel+ers|residents|individuals|victims)|prior (incidents|complaints|lawsuits)|"
                r"statistic|\bdata\b|monell|\btargets?\b|\btargeting\b|\bquotas?\b|interdiction")
CRIMINAL_SUSPICION_CUES = (r"shoplift|\bst(eal|ole|olen)\b|theft|accus|suspect|suspicion|search|detain|police|"
                           r"security|loss prevention|followed|receipt|\bbag\b")

def setting_code(value, level):
    """Normalize a model value such as 'T1 traffic stop' to its code ('T' at level 1, 'T1' at level 2)."""
    m = re.match(r"\s*([TPJCRISU])(\d)?\b", str(value or "").upper())
    if not m:
        return None
    if level == 1:
        return m.group(1)
    return m.group(1) + m.group(2) if m.group(2) else None


def check_setting_group(data, source_pages, page_texts, ocr=False):
    """Citation check for the setting fields, plus consistency rules between them."""
    data = data if isinstance(data, dict) else {}
    kw = dict(source_pages=source_pages, page_texts=page_texts, ocr=ocr)

    def normalized(item, level):
        item = dict(item) if isinstance(item, dict) else {"value": item}
        code = setting_code(item.get("value"), level)
        if code in (None, "U"):
            item.update(value=None, basis="not_stated")
        else:
            item["value"] = code
        return item

    records = []

    # Scope first: setting fields apply only to complaints about a profiling encounter
    sc = data.get("case_scope")
    sc = dict(sc) if isinstance(sc, dict) else {"value": sc}
    scope_value = str(sc.get("value") or "").strip().lower()
    if scope_value in CASE_SCOPES:
        sc["value"] = scope_value
    else:
        sc.update(value=None, basis="not_stated")
    scope = check_item("case_scope", "case_scope", sc, **kw)
    records.append(scope)

    # Pattern or practice alleged: "yes" needs a quote; absence cannot be verified, so anything else is not stated
    pa = data.get("pattern_alleged")
    pa = dict(pa) if isinstance(pa, dict) else {"value": pa}
    if scope["status"] == "verified" and scope["value"] == "employment":
        records.append({"field": "pattern_alleged", "field_type": "pattern_alleged", "value": None, "page": None,
                        "quote": None, "basis": "not_applicable", "status": "not stated",
                        "problems": "not applicable: case scope is employment"})
    elif str(pa.get("value") or "").strip().lower() == "yes":
        pa["value"] = "yes"
        records.append(check_item("pattern_alleged", "pattern_alleged", pa, **kw))
    else:
        records.append({"field": "pattern_alleged", "field_type": "pattern_alleged", "value": None, "page": None,
                        "quote": None, "basis": "not_stated", "status": "not stated", "problems": ""})
    if scope["status"] == "verified" and scope["value"] != "encounter":
        for f, t in [("primary_setting", "setting_l1"), ("primary_setting_detail", "setting_l2"),
                     ("traffic_stop", "traffic_stop"), ("criminal_suspicion", "criminal_suspicion")]:
            records.append({"field": f, "field_type": t, "value": None, "page": None, "quote": None,
                            "basis": "not_applicable", "status": "not stated",
                            "problems": f"not applicable: case scope is {scope['value']}"})
        return records

    primary = check_item("primary_setting", "setting_l1", normalized(data.get("primary_setting"), 1), **kw)
    records.append(primary)
    detail = check_item("primary_setting_detail", "setting_l2",
                        normalized(data.get("primary_setting_detail"), 2), **kw)
    if detail["status"] == "verified" and not (primary["status"] == "verified"
                                               and detail["value"].startswith(primary["value"])):
        detail.update(status="failed", problems="sub-code does not belong to the verified primary setting")
    records.append(detail)

    mentioned = []
    for i, item in enumerate(data.get("settings") or []):
        item = normalized(item, 1)
        if item["value"] is not None:
            mentioned.append(check_item(f"settings[{i}].code", "setting_l1", item, **kw))
    records += mentioned
    verified_codes = {r["value"] for r in [primary] + mentioned if r["status"] == "verified"}

    # Traffic stop: "yes" needs a quote describing a stop; "no" must be consistent with verified settings
    ts = data.get("traffic_stop")
    ts = dict(ts) if isinstance(ts, dict) else {"value": ts}
    v = str(ts.get("value") or "").strip().lower()
    if v == "yes":
        ts["value"] = "yes"
        records.append(check_item("traffic_stop", "traffic_stop", ts, **kw))
    else:
        rec = {"field": "traffic_stop", "field_type": "traffic_stop", "value": None, "page": None,
               "quote": None, "basis": "not_stated", "status": "not stated", "problems": ""}
        if v == "no":
            rec.update(value="no", basis="inferred")
            if verified_codes and "T" not in verified_codes:
                rec["status"] = "verified"
            else:
                rec.update(status="failed", problems="'no' requires verified settings, none of them a vehicle setting (T)")
        records.append(rec)

    # Criminal suspicion: applies only when the verified primary setting is commercial (C)
    cs = data.get("criminal_suspicion")
    cs = dict(cs) if isinstance(cs, dict) else {"value": cs}
    v = str(cs.get("value") or "").strip().lower()
    base = {"field": "criminal_suspicion", "field_type": "criminal_suspicion", "page": None, "quote": None,
            "problems": ""}
    if primary["status"] != "verified":
        records.append({**base, "value": None, "basis": "not_stated", "status": "not stated"})
    elif primary["value"] != "C":
        records.append({**base, "value": None, "basis": "not_applicable", "status": "not stated"})
    elif v == "present":
        cs["value"] = "present"
        records.append(check_item("criminal_suspicion", "criminal_suspicion", cs, **kw))
    elif v == "absent":
        records.append({**base, "value": "absent", "basis": "inferred", "status": "verified"})
    else:
        records.append({**base, "value": None, "basis": "not_stated", "status": "not stated"})
    return records


NAME_TYPES = {"plaintiff_name", "defendant_name", "case_name"}

def check_item(field, field_type, item, source_pages, page_texts, subject=None, ocr=False):
    """Run the citation checks on one extracted value and return an audit record.
    For OCR documents the quote match threshold is lowered (0.8 instead of 0.9)."""
    if not isinstance(item, dict):
        item = {"value": item, "page": None, "quote": None, "basis": "explicit"}
    value, page, quote = item.get("value"), item.get("page"), item.get("quote")
    basis = item.get("basis") or ("not_stated" if value in (None, "") else "explicit")
    record = {"field": field, "field_type": field_type, "value": value, "page": page,
              "quote": quote, "basis": basis}

    if value in (None, "") or basis == "not_stated":
        record.update(value=None, status="not stated", problems="")
        return record

    problems = []
    try:
        page = int(page)
    except (TypeError, ValueError):
        page = None
    page_ok = page in source_pages
    if not page_ok:
        problems.append(f"page {item.get('page')} was not among the retrieved pages {source_pages}")
    page_text = page_texts.get(page, "")
    match = quote_on_page(quote, page_text, min_ratio=0.8 if ocr else 0.9) if (quote and page) else None
    # Names: accept if the name itself is on the cited page, even when OCR errors spoil the full quote
    if not match and page and field_type in NAME_TYPES and _compact(value) and _compact(value) in _compact(page_text):
        match = "name on page"
    if not match:
        problems.append("quote not found on the cited page")
    supported = value_supported(field_type, value, quote)
    if not supported:
        problems.append("quote does not contain the value")
    if subject is not None and not mentions_subject(quote, subject, field_type, value):
        supported = False
        problems.append(f"quote does not refer to plaintiff {subject!r}")

    record.update(page=page, page_check=page_ok, quote_check=match or "not found",
                  support_check=supported,
                  status="verified" if (page_ok and match and supported) else "failed",
                  problems="; ".join(problems))

    # Gender basis is determined from the quote, not taken from the model's label
    if field_type == "gender" and record["status"] == "verified":
        words = set(re.findall(r"\w+", (quote or "").lower()))
        record["basis"] = "explicit" if EXPLICIT_GENDER_TERMS & words else "inferred"
    return record


def iter_fields(group_name, data):
    """Yield (field path, field type, item) for every value in a group's JSON output."""
    data = data if isinstance(data, dict) else {}
    if group_name == "parties":
        for i, p in enumerate(data.get("plaintiffs") or []):
            p = p if isinstance(p, dict) else {}
            name_item = p.get("name")
            name = name_item.get("value") if isinstance(name_item, dict) else name_item
            yield f"plaintiffs[{i}].name", "plaintiff_name", name_item, None
            yield f"plaintiffs[{i}].age", "age", p.get("age"), name or ""
            yield f"plaintiffs[{i}].gender", "gender", p.get("gender"), name or ""
            yield f"plaintiffs[{i}].race", "race", p.get("race"), name or ""
        for j, d in enumerate(data.get("defendants") or []):
            d = d if isinstance(d, dict) else {}
            yield f"defendants[{j}].name", "defendant_name", d.get("name"), None
        return
    for key in ("case_number", "case_name", "date_filed", "document_title"):
        yield key, ("case_name" if key == "document_title" else key), data.get(key), None


def check_extraction(group_name, data, source_pages, page_texts, ocr=False):
    if group_name == "setting":
        return check_setting_group(data, source_pages, page_texts, ocr=ocr)
    return [check_item(f, t, item, source_pages, page_texts, subject=subj, ocr=ocr)
            for f, t, item, subj in iter_fields(group_name, data)]

## Extraction with Citation Check and Automatic Correction

`extract_group` runs the full extraction for one field group:

1. **Retrieve:** collects the group's excerpts and notes which pages they come from.
2. **Extract:** sends the excerpts and the group's instructions to the model, which returns JSON.
3. **Check:** runs the citation check on every value.
4. **Correct:** if any value fails, sends it back to the model once, with the reason it failed, and asks it either to fix the page and quote or to mark the field "not stated." The corrected answer is checked again; values that still fail are flagged for review.

Each run prints a one-line summary (pages used, and the number of values verified, failed and not stated, plus corrections) and returns every value with its check results for the audit table.

**Gender follow-up.** Complaints often refer to a plaintiff only with pronouns, and in a long complaint the sentences that pair a pronoun with the plaintiff's name may not be retrieved by the general queries. For each plaintiff whose gender was not verified, `followup_gender` retrieves passages using that plaintiff's name and asks again. The answer passes through the same citation check, including the subject and proximity rules, and replaces the original only if verified, so the follow-up can add a supported value but never weakens a check.


In [ ]:
SETTING_CORRECTION_HINT = (
    "For setting codes, the quote must NAME the place or the kind of encounter "
    "(e.g., a street or sidewalk, a home, a store, a vehicle stop, an airport, a police station), "
    "not only describe the interaction. ")


def extract_group(group_name, retriever, chunks, page_texts, model=None, max_corrections=1, verbose=True, ocr=False):
    """Retrieve excerpts for a field group, extract JSON, run the citation check,
    and ask the model to correct failed values (once by default)."""
    model = model or EXTRACTION_MODEL
    group = FIELD_GROUPS[group_name]
    docs = retrieve_for_group(retriever, chunks, group)
    source_pages = sorted({d.metadata["page"] for d in docs})

    user_prompt = f"### Context excerpts\n{format_excerpts(docs)}\n\n### Task\n{group['instructions']}"
    data = response_json(user_prompt, EXTRACTION_SYSTEM, model=model)
    records = check_extraction(group_name, data, source_pages, page_texts, ocr=ocr)

    corrections = 0
    while corrections < max_corrections and any(r["status"] == "failed" for r in records):
        failed = [f"- {r['field']} = {r['value']!r}: {r['problems']}" for r in records if r["status"] == "failed"]
        if verbose:
            print(f"[{group_name}] citation check failed for {len(failed)} value(s) - requesting correction")
        correction_prompt = (
            f"{user_prompt}\n\n### Citation check failed\n"
            "These values could not be verified against the excerpts:\n" + "\n".join(failed) +
            f"\n\nFor each one, either correct the page and copy the quote character-for-character from an excerpt "
            f"labeled with that page, or set the field to not_stated if the excerpts do not support it. "
            f"Allowed pages: {source_pages}. "
            + (SETTING_CORRECTION_HINT if group_name == "setting" else "")
            + "Return the complete JSON again.\n\n"
            f"### Previous answer\n{json.dumps(data, ensure_ascii=False)}"
        )
        data = response_json(correction_prompt, EXTRACTION_SYSTEM, model=model)
        records = check_extraction(group_name, data, source_pages, page_texts, ocr=ocr)
        corrections += 1

    for r in records:
        r.update(group=group_name, corrections=corrections, retrieved_pages=source_pages)
    if verbose:
        n_ok = sum(r["status"] == "verified" for r in records)
        n_fail = sum(r["status"] == "failed" for r in records)
        print(f"[{group_name}] pages {source_pages} | verified {n_ok} | failed {n_fail} | "
              f"not stated {len(records) - n_ok - n_fail} | corrections {corrections}")
    return data, records


# ---- Gender follow-up: name-based retrieval for plaintiffs whose gender was not verified ----
GENDER_FOLLOWUP_TASK = """Determine the gender of the plaintiff named {name}.
- Prefer a quote with an explicit term (man, woman, male, female).
- If the complaint refers to this plaintiff only with pronouns or honorifics, quote ONE sentence in which the pronoun or
  honorific appears together with "Plaintiff" or the plaintiff's name (e.g. "Plaintiff told the officers that she owned
  the gun", "Ms. {surname} was handcuffed"), and set basis to "inferred".
- Never infer gender from the name itself. If no such sentence appears in the excerpts, return not_stated.

Return JSON exactly in this shape:
{{"gender": FIELD}}
where FIELD = {{"value": ..., "page": P, "quote": "...", "basis": "explicit" | "inferred" | "not_stated"}}"""


def followup_gender(retriever, chunks, page_texts, records, model=None, verbose=True, ocr=False,
                    per_query=4, top_k=10):
    """For each plaintiff whose gender was not verified, retrieve passages that name that plaintiff and ask again.
    The answer goes through the same citation check (including the subject/proximity check); it replaces the
    original record only if verified, so a follow-up can add a supported value but never weaken a check."""
    model = model or EXTRACTION_MODEL
    out = list(records)
    for k, rec in enumerate(out):
        m = re.match(r"plaintiffs\[(\d+)\]\.gender$", rec["field"])
        if not m or rec["status"] == "verified":
            continue
        name_rec = next((r for r in out if r["field"] == f"plaintiffs[{m.group(1)}].name"), None)
        name = name_rec["value"] if name_rec and isinstance(name_rec.get("value"), str) else None
        if not name:
            continue
        surname = name.split()[-1]
        queries = [f"{name} she her woman female", f"{name} he his man male",
                   f"Plaintiff {surname} Ms. Mrs. Mr.", f"Plaintiff {surname} she he her his"]
        seen, docs = set(), []
        for q in queries:
            for d in retriever.invoke(q)[:per_query]:
                if d.metadata["chunk_id"] not in seen:
                    seen.add(d.metadata["chunk_id"])
                    docs.append(d)
        docs = sorted(docs[:top_k], key=lambda d: (d.metadata["page"], d.metadata["chunk_id"]))
        source_pages = sorted({d.metadata["page"] for d in docs})
        prompt = (f"### Context excerpts\n{format_excerpts(docs)}\n\n### Task\n"
                  + GENDER_FOLLOWUP_TASK.format(name=name, surname=surname))
        data = response_json(prompt, EXTRACTION_SYSTEM, model=model)
        item = data.get("gender") if isinstance(data, dict) else None
        new = check_item(rec["field"], "gender", item, source_pages, page_texts, subject=name, ocr=ocr)
        if new["status"] == "verified":
            new.update(group=rec.get("group", "parties"), corrections=rec.get("corrections", 0),
                       retrieved_pages=source_pages, followup="gender")
            out[k] = new
        else:
            out[k] = {**rec, "followup": "gender: no supported quote found"}
        if verbose:
            print(f"[gender follow-up] {name}: {new['status']}"
                  + (f" = {new['value']} ({new['basis']})" if new["status"] == "verified" else ""))
    return out


## Assembling the Output

`complaint_df` has one row per plaintiff. Case-level fields repeat on each row, and all defendants are listed in one cell. Each value appears only if it passed the citation check:

- **a value**: verified against the complaint;
- **"Not stated"**: the complaint, as retrieved, does not state it;
- **empty (None)**: the model's value failed the citation check, and the field is listed under *Needs review*.

*Inferred* lists values that rest on indirect evidence, such as gender from pronouns. `audit_df` keeps every value with its page, quote and check results.

Two document-level checks are read directly from the page text, independent of the model's quotes:
- **Judge initials** come from the case-number suffix printed in the court's ECF headers (e.g., `2:23-cv-01234-ABC-DEF`), so they no longer depend on which quote the model chooses.
- **Defendant completeness:** names introduced in the complaint as "Defendant X" (and "Does 1 through 50" placeholders) are compared with the extracted defendants; any that do not match are listed under *Possible missing defendants* for review. Matching uses shared name words and acronyms (e.g., "RPD" for "Riverton Police Department"), so the list may include false alarms; it is a prompt to check, not an error.


In [ ]:
SETTING_COLUMNS = ["Case scope", "Setting", "Setting detail", "All settings", "Traffic stop", "Pattern alleged", "Criminal suspicion"]
OUTPUT_COLUMNS = ["Case number", "Case name", "Date filed", "Plaintiff name", "Plaintiff age",
                  "Plaintiff gender", "Plaintiff race", "Defendant name", "Document"] + SETTING_COLUMNS


def _cell(rec):
    """Value to show in complaint_df for one audit record."""
    if rec is not None and rec.get("basis") == "not_applicable":
        return "Not applicable"
    if rec is None or rec["status"] == "not stated":
        return "Not stated"
    if rec["status"] != "verified":
        return None
    if rec["field_type"] == "date_filed":
        d = parse_date(rec["value"])
        return d.isoformat() if d else rec["value"]
    return rec["value"]


def assemble_rows(records, source_file):
    by_field = {r["field"]: r for r in records}
    n_plaintiffs = max([int(m.group(1)) + 1 for f in by_field
                        for m in [re.match(r"plaintiffs\[(\d+)\]", f)] if m] or [1])
    defendant_recs = [r for f, r in by_field.items() if f.startswith("defendants[")]
    verified_defendants = [r["value"] for r in defendant_recs if r["status"] == "verified"]

    # Setting fields are case-level: computed once, repeated on each plaintiff row
    def labeled(key, table):
        rec = by_field.get(key)
        value = _cell(rec)
        return f"{value} – {table[value]}" if rec and rec["status"] == "verified" and value in table else value
    setting_recs = [r for f, r in by_field.items()
                    if f.startswith("settings[") or f in ("case_scope", "primary_setting", "primary_setting_detail",
                                                           "traffic_stop", "pattern_alleged", "criminal_suspicion")]
    all_settings = sorted({r["value"] for f, r in by_field.items()
                           if (f.startswith("settings[") or f == "primary_setting") and r["status"] == "verified"})
    setting_values = {
        "Case scope": labeled("case_scope", CASE_SCOPES),
        "Setting": labeled("primary_setting", SETTING_LEVEL1),
        "Setting detail": labeled("primary_setting_detail", SETTING_LEVEL2),
        "All settings": "; ".join(all_settings) if all_settings else "Not stated",
        "Traffic stop": _cell(by_field.get("traffic_stop")),
        "Pattern alleged": _cell(by_field.get("pattern_alleged")),
        "Criminal suspicion": _cell(by_field.get("criminal_suspicion")),
    }

    rows = []
    for i in range(n_plaintiffs):
        fields = {
            "Case number": "case_number", "Case name": "case_name", "Date filed": "date_filed",
            "Document": "document_title",
            "Plaintiff name": f"plaintiffs[{i}].name", "Plaintiff age": f"plaintiffs[{i}].age",
            "Plaintiff gender": f"plaintiffs[{i}].gender", "Plaintiff race": f"plaintiffs[{i}].race",
        }
        row = {col: _cell(by_field.get(key)) for col, key in fields.items()}
        row["Defendant name"] = "; ".join(verified_defendants) if verified_defendants else (
            None if defendant_recs else "Not stated")

        row.update(setting_values)
        relevant = [by_field[k] for k in fields.values() if k in by_field] + defendant_recs + setting_recs
        row["Needs review"] = ", ".join(r["field"] for r in relevant if r["status"] == "failed")
        row["Inferred"] = ", ".join(r["field"] for r in relevant
                                    if r["status"] == "verified" and r["basis"] == "inferred")
        row["Source file"] = source_file
        rows.append(row)
    return rows


# ---- Document-level checks read directly from the page text (deterministic, no model involved) ----

# Full case number with judge suffix, as printed in ECF headers, e.g. "2:23-cv-01234-ABC-DEF"
FULL_CASE_NUMBER_RE = re.compile(r"(?i:\b\d{1,2}:\d{2}-[a-z]{2,4}-\d{3,6})((?:-[A-Z]{1,4})+)\b")

def find_judge_initials(page_texts):
    """Judge initials from the case-number suffix printed in the ECF headers (most common across pages)."""
    found = Counter(m.group(1).lstrip("-") for t in page_texts.values() for m in FULL_CASE_NUMBER_RE.finditer(t))
    return found.most_common(1)[0][0] if found else None


# Defendant completeness: names introduced as "Defendant X" and Doe placeholders anywhere in the complaint
DEFENDANT_MENTION_RE = re.compile(
    r"\bDefendants?,?[ \t]+(?:the[ \t]+)?"
    r"(?:(?:Officer|Sergeant|Sgt\.|Deputy|Detective|Lieutenant|Lt\.|Chief|Trooper|Agent|Captain|Capt\.)[ \t]+)?"
    r"([A-Z][A-Za-z.'’-]+(?:[ \t]+(?:of[ \t]+)?[A-Z][A-Za-z.'’-]+){0,5})")
DOE_MENTION_RE = re.compile(r"\b(?:DOES|Does)[ \t]+\d+[ \t]*(?:through|to|-|–)[ \t]*\d+")
GENERIC_NAME_WORDS = {"city", "county", "state", "police", "department", "officer", "officers", "deputy",
                      "deputies", "sheriff", "defendant", "defendants", "the", "and"}
NON_NAMES = {"The", "This", "That", "These", "Those", "Each", "All", "Such", "In", "On", "At", "Upon",
             "Plaintiff", "Plaintiffs", "Count", "Counts", "As", "Any"}

def _initials(name):
    return "".join(w[0] for w in re.findall(r"[A-Za-z]+", name) if w.lower() not in {"of", "the", "and"}).upper()

def _matches_extracted(candidate, extracted_names):
    """Does a candidate mention correspond to an extracted defendant (shared name word or acronym)?"""
    if DOE_MENTION_RE.search(candidate):
        return any(re.search(r"\bdoes?\b", n, re.I) for n in extracted_names)
    words = {w.lower() for w in re.findall(r"[A-Za-z]{3,}", candidate)} - GENERIC_NAME_WORDS
    if not words:
        return True                                  # only generic words (e.g. "Police Department"): not checkable
    for name in extracted_names:
        if words & {w.lower() for w in re.findall(r"[A-Za-z]{3,}", name)}:
            return True
        if candidate.replace(".", "").isupper() and candidate.replace(".", "") == _initials(name):
            return True                              # acronym, e.g. "RPD"
    return False

def find_possible_missing_defendants(page_texts, extracted_names, limit=10):
    """Defendants mentioned in the complaint that do not match any extracted defendant (flagged for review)."""
    candidates = []
    for text in page_texts.values():
        candidates += [m.group(1).strip(" .,;") for m in DEFENDANT_MENTION_RE.finditer(text)]
        candidates += [m.group(0) for m in DOE_MENTION_RE.finditer(text)]
    missing, seen = [], set()
    for c in candidates:
        if c.split()[0] in NON_NAMES or c.lower() in seen or re.search(r"\bplaintiffs?\b", c, re.I):
            continue
        seen.add(c.lower())
        if not _matches_extracted(c, extracted_names):
            missing.append(c)
    return missing[:limit]


def process_complaint(pdf_path, model=None, verbose=True):
    """Full pipeline for one complaint: load & clean -> index -> extract (3 groups) -> citation check -> tables."""
    source_file = os.path.basename(pdf_path)
    page_docs, page_texts, _ = load_complaint(pdf_path)
    pages_total = len(page_docs)
    if MAX_PAGES_INDEXED and pages_total > MAX_PAGES_INDEXED:     # long file: index the first pages only
        page_docs = page_docs[:MAX_PAGES_INDEXED]
        page_texts = {k: v for k, v in page_texts.items() if k <= MAX_PAGES_INDEXED}
        if verbose:
            print(f"Long file: indexing pages 1-{MAX_PAGES_INDEXED} of {pages_total}")
    ocr_pages = detect_ocr_pages(pdf_path)
    ocr_document = len(ocr_pages) >= 0.5 * len(page_docs)       # mostly scanned pages
    if verbose and ocr_document:
        print(f"OCR document ({len(ocr_pages)} of {len(page_docs)} pages scanned): tolerant quote matching")
    chunks, vectorstore, retriever = build_index(page_docs)
    try:
        _, case_records = extract_group("case", retriever, chunks, page_texts, model=model,
                                        verbose=verbose, ocr=ocr_document)
        _, party_records = extract_group("parties", retriever, chunks, page_texts, model=model,
                                         verbose=verbose, ocr=ocr_document)
        party_records = followup_gender(retriever, chunks, page_texts, party_records, model=model,
                                        verbose=verbose, ocr=ocr_document)
        _, setting_records = extract_group("setting", retriever, chunks, page_texts, model=model,
                                           verbose=verbose, ocr=ocr_document)
    finally:
        vectorstore.delete_collection()          # keep complaints separate in batch runs

    records = case_records + party_records + setting_records

    # Document-level checks from the page text
    judge = find_judge_initials(page_texts)
    defendant_names = [r["value"] for r in party_records
                       if r["field"].startswith("defendants[") and isinstance(r["value"], str)]
    missing = find_possible_missing_defendants(page_texts, defendant_names)
    if verbose and missing:
        print(f"Possible missing defendants (review): {'; '.join(missing)}")

    complaint_df = pd.DataFrame(assemble_rows(records, source_file))
    complaint_df["OCR document"] = ocr_document
    complaint_df["Pages indexed"] = f"{len(page_docs)} of {pages_total}"
    complaint_df["Possible missing defendants"] = "; ".join(missing)
    audit_df = pd.DataFrame(records)
    audit_df.insert(0, "source_file", source_file)
    audit_df["pages_indexed"], audit_df["pages_total"] = len(page_docs), pages_total
    audit_df["ocr_document"] = ocr_document
    audit_df["judge_initials"] = judge
    audit_df["possible_missing_defendants"] = "; ".join(missing)
    return complaint_df, audit_df

## Extracting Data from the Complaint

In [ ]:
complaint_df, audit_df = process_complaint(PDF_PATH)
complaint_df[OUTPUT_COLUMNS + ["Needs review", "Inferred"]]

## Citation Audit

Each extracted value with its cited page, the supporting quote, and the result of each check. Review any row with status "failed", and spot-check a few "verified" rows against the PDF.

In [ ]:
pd.set_option("display.max_colwidth", 120)
audit_df[["field", "value", "basis", "page", "quote", "page_check", "quote_check",
          "support_check", "status", "corrections", "problems"]]

In [ ]:
# Summary of the citation check
def citation_summary(audit):
    """Actionable summary of the citation check for one complaint (or a whole batch)."""
    total = len(audit)
    counts = audit["status"].value_counts()
    verified, failed, absent = (int(counts.get(s, 0)) for s in ["verified", "failed", "not stated"])
    stated = verified + failed

    print("CITATION CHECK SUMMARY")
    print(f"  Values extracted: {total}")
    print(f"  Verified:         {verified} of {stated} stated values"
          + (f" ({verified / stated:.0%})" if stated else ""))
    print(f"  Failed:           {failed}")
    not_applicable = int((audit["basis"] == "not_applicable").sum()) if "basis" in audit else 0
    print(f"  Not stated:       {absent - not_applicable}"
          + (f" (plus {not_applicable} not applicable)" if not_applicable else ""))

    # Correction rounds are recorded per field group, not per value
    groups = audit.groupby("group")["corrections"].max()
    corrected = groups[groups > 0].index.tolist()
    print(f"  Groups that needed a correction round: {', '.join(corrected) if corrected else 'none'}")

    # 1. Values that failed: must be reviewed
    print("\n1. NEEDS REVIEW (failed the citation check; left blank in complaint_df)")
    rows = audit[audit["status"] == "failed"]
    if rows.empty:
        print("   None.")
    for _, r in rows.iterrows():
        print(f"   - {r['field']} = {r['value']!r}: {r['problems']}")

    # 2. Values reported as absent: cannot be verified automatically
    print("\n2. SPOT-CHECK (reported as not stated; search the PDF to confirm)")
    rows = audit[(audit["status"] == "not stated") & (audit["basis"] != "not_applicable")]
    if rows.empty:
        print("   None.")
    for _, r in rows.iterrows():
        print(f"   - {r['field']}")

    # 3. Verified values that rest on indirect evidence
    print("\n3. INFERRED (verified, but based on indirect evidence: pronouns, a signature date,")
    print("   or the absence of a vehicle or commercial setting)")
    rows = audit[(audit["status"] == "verified") & (audit["basis"] == "inferred")]
    if rows.empty:
        print("   None.")
    for _, r in rows.iterrows():
        if pd.notna(r["page"]) and r["quote"]:
            print(f"   - {r['field']} = {r['value']!r} (p. {int(r['page'])}): \"{r['quote']}\"")
        else:
            print(f"   - {r['field']} = {r['value']!r} (derived from the verified settings, no quote)")

    # 4. Defendants mentioned in the complaint but not extracted
    print("\n4. POSSIBLE MISSING DEFENDANTS (mentioned in the complaint, not extracted)")
    missing = (audit.groupby("source_file")["possible_missing_defendants"].first()
               if "possible_missing_defendants" in audit else pd.Series(dtype=str))
    missing = missing[missing.fillna("").astype(str).str.len() > 0]
    if missing.empty:
        print("   None.")
    for src, names in missing.items():
        print(f"   - {src}: {names}")

    # Overall verdict
    issues = []
    if failed:
        issues.append(f"review {failed} failed value(s)")
    if not missing.empty:
        issues.append("check the defendant list")
    print("\nVERDICT:", "Ready for analysis." if not issues else "Before using: " + "; ".join(issues) + ".")

citation_summary(audit_df)

## Building Analysis-Ready Tables

The citation audit is in **long format** (one row per extracted value), which is ideal for verification but not for analysis. This step reshapes it into a small relational dataset linked by `case_id`:

```
cases_df       (1 row per complaint)
   │ case_id
   ├── plaintiffs_df  (1 row per plaintiff)
   ├── defendants_df  (1 row per defendant)
   └── evidence_df    (1 row per extracted value: page + quote)
quality_df     (share of verified / failed / not stated values per attribute)
```


- **`cases_df`**: filing date and year, whether the document is an amended complaint, and the judge's initials from the case number.
- **`plaintiffs_df`**: name, age, gender and race, with an `all_verified` flag and race coded with the scheme below.
- **`defendants_df`**: whether each defendant is an entity, an individual, or a "Doe" placeholder, and for named individuals, whether they are sued individually, in an official capacity, or both.
- **`evidence_df`**: the page and supporting quote for every value, so any figure can be traced to its source.
- **`quality_df`**: the share of verified, failed and not-stated values for each attribute.

Standardized columns (`*_clean`, `*_std`, `age_num`) are added **next to** the values as printed, so coding decisions stay visible and reversible. `case_id` is the base case number (e.g., `2:23-cv-01234`), so the same case matches across sources; the full number is kept in `case_number`.

**Race coding scheme.** Race and ethnicity are coded with a multi-label scheme based on the U.S. Census race groups, with Hispanic/Latino treated as a category:
- `race_categories` lists every category the description matches (e.g., "Black; Hispanic/Latino"), and `race_multiple` flags descriptions with more than one.
- `race_std` is the single category, or "Multiple"; "Person of color (unspecified)" when the complaint says only "person of color" or "non-white"; and "Unmapped" when nothing matches (review these and extend `RACE_CATEGORIES`).
- `race_person_of_color` flags any non-White category or an explicit "person of color"/"non-white" description.
- `race_other_descriptors` records religious or other identity terms (e.g., "Jewish") that are not race categories.

For example, "Afro-Latina" is coded under both Black and Hispanic/Latino, with `race_multiple` set to True; "a person of color of Jewish descent" is coded as person of color (unspecified), with "Jewish" recorded as an other descriptor.

**Defendant types.** Defendants are classified as **Entity** (government bodies, agencies, companies), **Individual**, or **Doe** (placeholders such as "Does 1 through 50", which should usually be excluded from counts). Capacity (`sued_individually`, `sued_official`) is recorded only for named individuals and only when the quote states it; otherwise it is `NaN` (unknown). Each attribute keeps its basis and verification status, so analyses can be restricted to verified, explicit values. The same function works on the combined audit of many complaints (see Batch Processing).


In [ ]:
# ---- Race/ethnicity coding scheme (multi-label) ----
# Categories follow the U.S. Census race groups, with Hispanic/Latino treated as a category, as is common
# in research on policing. A description can match several categories (e.g., "Black Latino").
RACE_CATEGORIES = {
    "Black": r"\bblack\b|african[- ]american|\bafro[- ]",
    "Hispanic/Latino": (r"hispanic|latin[oaex]\b|latinx|chican[oa]|mexican|puerto ric|cuban|dominican|"
                        r"colombian|salvador|guatemal|hondur|venezuel|peruvian|ecuador"),
    "White": r"(?<!non-)(?<!non )\bwhite\b|caucasian",
    "Asian": r"\basian\b|chinese|japanese|korean|vietnamese|filipin|south asian|indian[- ]american",
    "Native American": r"native american|american indian|alaska native|indigenous",
    "Pacific Islander": r"pacific islander|native hawaiian|samoan|tongan",
    "Middle Eastern/North African": r"middle eastern|\barab|north african|persian|iranian",
}
PERSON_OF_COLOR_PATTERN = r"person of colou?r|people of colou?r|\bpoc\b|non-?\s?white|\bminority\b"
OTHER_DESCRIPTORS = ["jewish", "muslim", "sikh", "hindu", "immigrant"]   # recorded, not coded as race


def code_race(text):
    """Code a race/ethnicity description into categories; the original text is always kept."""
    if not isinstance(text, str) or not text.strip():
        return {"race_categories": None, "race_std": None, "race_multiple": None,
                "race_person_of_color": None, "race_other_descriptors": None}
    t = text.lower()
    cats = [c for c, pattern in RACE_CATEGORIES.items() if re.search(pattern, t)]
    poc = bool(re.search(PERSON_OF_COLOR_PATTERN, t)) or bool(set(cats) - {"White"})
    other = [w.title() for w in OTHER_DESCRIPTORS if re.search(rf"\b{w}\b", t)]
    if len(cats) == 1:
        std = cats[0]
    elif len(cats) > 1:
        std = "Multiple"
    elif poc:
        std = "Person of color (unspecified)"
    else:
        std = "Unmapped"
    return {"race_categories": "; ".join(cats) or None, "race_std": std, "race_multiple": len(cats) > 1,
            "race_person_of_color": poc, "race_other_descriptors": "; ".join(other) or None}


# ---- Defendant types and capacity ----
DOE_PATTERN = r"^\s*(?:john|jane|richard)?\s*does?\b|\bunknown (?:officers?|defendants?)\b"
ENTITY_PATTERN = (r"county|city|town|village|township|state of|department|sheriff|police|"
                  r"board|agency|authority|commission|district|\binc\b|llc|corp|company")
INDIVIDUAL_CAPACITY = r"\bindividually\b|individual capacit|personal capacit"
OFFICIAL_CAPACITY = r"official capacit"

# ---- Case numbers: base number for joins, judge initials from the full number ----
CASE_BASE_RE = re.compile(r"\d{1,2}:\d{2}-[a-z]{2,4}-\d{3,6}", re.I)
JUDGE_SUFFIX_RE = re.compile(r"(?i:\d{1,2}:\d{2}-[a-z]{2,4}-\d{3,6})((?:-[A-Z]{1,4})+)")


def tidy_name(name):
    """Title-case names printed entirely in capitals (caption style); leave others unchanged."""
    return name.title() if isinstance(name, str) and name.isupper() else name


def build_analysis_tables(audit):
    """Reshape the long citation audit (one or many complaints) into analysis-ready tables."""
    audit = audit.reset_index(drop=True)
    parts = audit["field"].str.extract(
        r"^(?:(?P<entity>plaintiffs|defendants|settings)\[(?P<idx>\d+)\]\.)?(?P<attribute>\w+)$")
    audit = audit.drop(columns=[c for c in ["entity", "idx", "attribute"] if c in audit]).join(parts)
    audit["entity"] = audit["entity"].fillna("case")
    audit["idx"] = pd.to_numeric(audit["idx"])

    # ---- cases_df: one row per complaint ----
    case_rows = audit[audit["entity"] == "case"]
    cases = (case_rows.pivot(index="source_file", columns="attribute", values="value").reset_index())
    for col in ["case_number", "case_name", "date_filed", "document_title"]:
        if col not in cases:
            cases[col] = None

    # case_id = base number (e.g. 2:23-cv-01234), so the same case matches across sources
    def base_number(v):
        m = CASE_BASE_RE.search(str(v)) if isinstance(v, str) else None
        return m.group(0).lower() if m else v
    cases["case_id"] = cases["case_number"].map(base_number)

    # judge initials: from the ECF headers in the page text; quotes are a fallback
    page_judges = (audit.groupby("source_file")["judge_initials"].first()
                   if "judge_initials" in audit else pd.Series(dtype=object))

    def judge_initials(source_file):
        # preferred: read from the ECF headers in the page text (set by process_complaint)
        if pd.notna(page_judges.get(source_file)):
            return page_judges[source_file]
        # fallback: any case-level value or quote containing the full case number
        texts = case_rows.loc[case_rows["source_file"] == source_file, ["value", "quote"]].astype(str)
        for t in texts.values.ravel():
            m = JUDGE_SUFFIX_RE.search(t)
            if m:
                return m.group(1).lstrip("-")
        return None
    cases["judge_initials"] = cases["source_file"].map(judge_initials)

    cases["date_filed"] = pd.to_datetime(cases["date_filed"].map(parse_date), errors="coerce")
    cases["filing_year"] = cases["date_filed"].dt.year.astype("Int64")
    cases["amended"] = cases["document_title"].str.contains("AMENDED", case=False, na=False)
    ocr = (audit.groupby("source_file")["ocr_document"].first()
           if "ocr_document" in audit else pd.Series(dtype=bool))
    cases["ocr_document"] = cases["source_file"].map(ocr)

    # Encounter setting (verified values only; see the setting codebook)
    def verified(attr):
        rows = case_rows[(case_rows["attribute"] == attr) & (case_rows["status"] == "verified")]
        return rows.groupby("source_file")["value"].first()
    cases["case_scope"] = cases["source_file"].map(verified("case_scope"))
    cases["setting"] = cases["source_file"].map(verified("primary_setting"))
    cases["setting_label"] = cases["setting"].map(SETTING_LEVEL1)
    cases["setting_detail"] = cases["source_file"].map(verified("primary_setting_detail"))
    cases["setting_detail_label"] = cases["setting_detail"].map(SETTING_LEVEL2)
    setting_rows = audit[((audit["entity"] == "settings") | (audit["attribute"] == "primary_setting"))
                         & (audit["status"] == "verified")]
    cases["all_settings"] = cases["source_file"].map(
        setting_rows.groupby("source_file")["value"].apply(lambda v: "; ".join(sorted(set(v)))))
    cases["traffic_stop"] = cases["source_file"].map(verified("traffic_stop"))
    cases["pattern_alleged"] = cases["source_file"].map(verified("pattern_alleged"))
    cases["criminal_suspicion"] = cases["source_file"].map(verified("criminal_suspicion"))
    missing = (audit.groupby("source_file")["possible_missing_defendants"].first()
               if "possible_missing_defendants" in audit else pd.Series(dtype=object))
    cases["possible_missing_defendants"] = cases["source_file"].map(missing)
    cases = cases[["case_id", "case_name", "date_filed", "filing_year", "document_title",
                   "amended", "judge_initials", "ocr_document",
                   "case_scope", "setting", "setting_label", "setting_detail", "setting_detail_label", "all_settings",
                   "traffic_stop", "pattern_alleged", "criminal_suspicion", "possible_missing_defendants",
                   "case_number", "source_file"]]
    cases.columns.name = None
    keys = cases[["source_file", "case_id"]]

    def entity_table(entity):
        sub = audit[audit["entity"] == entity]
        if sub.empty:
            return pd.DataFrame()
        wide = sub.pivot(index=["source_file", "idx"], columns="attribute",
                         values=["value", "basis", "status", "page"])
        wide.columns = [a if m == "value" else f"{a}_{m}" for m, a in wide.columns]
        wide = wide.reset_index()
        wide["idx"] = wide["idx"].astype(int)
        return keys.merge(wide, on="source_file")

    # ---- plaintiffs_df: one row per plaintiff ----
    p = entity_table("plaintiffs")
    if len(p):
        p["name_clean"] = p["name"].map(tidy_name)
        p["age_num"] = pd.to_numeric(p["age"], errors="coerce")
        p["gender_std"] = p["gender"].map(lambda g: str(g).title() if isinstance(g, str) else g)
        p = p.join(pd.DataFrame([code_race(r) for r in p["race"]], index=p.index))
        status_cols = [c for c in ["name_status", "age_status", "gender_status", "race_status"] if c in p]
        p["all_verified"] = ~p[status_cols].eq("failed").any(axis=1)
        p = p[["case_id", "idx", "name_clean", "age_num", "gender_std", "gender_basis",
               "race", "race_std", "race_categories", "race_multiple", "race_person_of_color",
               "race_other_descriptors", "race_basis", "race_page", "all_verified", "name", "source_file"]]

    # ---- defendants_df: one row per defendant ----
    d = entity_table("defendants")
    if len(d):
        quotes = audit[audit["entity"] == "defendants"][["source_file", "idx", "quote"]]
        d = d.merge(quotes, on=["source_file", "idx"], how="left")
        d["name_clean"] = d["name"].map(tidy_name)
        is_doe = d["name"].str.contains(DOE_PATTERN, case=False, regex=True, na=False)
        is_entity = d["name"].str.contains(ENTITY_PATTERN, case=False, regex=True, na=False)
        d["type"] = np.select([is_doe, is_entity], ["Doe", "Entity"], default="Individual")
        # Capacity is recorded only for named individuals, and only when the quote states it;
        # otherwise it is unknown (NaN), not False ("an individual" describes a person, not a capacity)
        stated = d["quote"].str.contains(r"capacit|individually", case=False, na=False)
        known = (d["type"] == "Individual") & stated
        d["sued_individually"] = d["quote"].str.contains(INDIVIDUAL_CAPACITY, case=False, na=False).where(known)
        d["sued_official"] = d["quote"].str.contains(OFFICIAL_CAPACITY, case=False, na=False).where(known)
        d = d[["case_id", "idx", "name_clean", "type", "sued_individually", "sued_official",
               "name_status", "name", "source_file"]]

    # ---- evidence_df: the audit with case_id ----
    evidence = keys.merge(audit, on="source_file")[
        ["case_id", "field", "value", "basis", "page", "quote", "status", "problems", "source_file"]]

    # ---- quality_df: share of verified / failed / not stated per attribute ----
    prefix = audit["entity"].map({"plaintiffs": "plaintiff_", "defendants": "defendant_",
                                  "settings": "setting_mentioned_", "case": ""})
    quality = (audit.assign(attribute=prefix + audit["attribute"])
               .groupby("attribute")["status"].value_counts(normalize=True)
               .unstack(fill_value=0).round(2))
    quality = quality.reindex(columns=["verified", "failed", "not stated"], fill_value=0.0)

    return {"cases": cases, "plaintiffs": p, "defendants": d, "evidence": evidence, "quality": quality}


tables = build_analysis_tables(audit_df)
cases_df, plaintiffs_df, defendants_df = tables["cases"], tables["plaintiffs"], tables["defendants"]
evidence_df, quality_df = tables["evidence"], tables["quality"]

### `cases_df`: one row per complaint

In [ ]:
cases_df

**Example (André and English v. Clayton County):**

`judge_initials` is read from the case-number suffix in the ECF headers (`-MHC`; some cases list two judges, e.g. `ABC-DEF`), which identifies the assigned judge(s), not the court. `ocr_document` flags scanned complaints. `amended` flags amended complaints, whose filing date differs from the date the case was opened. `possible_missing_defendants` lists defendants mentioned in the complaint but not extracted.

The setting columns (`setting`, `setting_detail`, `all_settings`, `traffic_stop`, `criminal_suspicion`) contain verified values only; codes are explained in the setting codebook, with labels in `setting_label` and `setting_detail_label`. For the test complaint, the expected coding is J – Travel hub or border, J1 – Airport or air travel, traffic stop: no.


### `plaintiffs_df`: one row per plaintiff

In [ ]:
plaintiffs_df

`age_num` is numeric, with `NaN` when the age is not stated, so averages skip missing values. `race` keeps the complaint's own term; `race_std`, `race_categories`, `race_person_of_color` and `race_other_descriptors` follow the race coding scheme above, and "Unmapped" means `RACE_CATEGORIES` needs a new pattern. `all_verified` is False if any stated value for the plaintiff failed the citation check.

### `defendants_df`: one row per defendant

In [ ]:
defendants_df

`type` is a keyword-based first pass (government bodies, agencies and companies are "Entity"; placeholders such as "Does 1 through 50" are "Doe"); check a sample for misclassifications. Capacity is `NaN` for entities, Doe placeholders, and individuals whose quote does not state a capacity (e.g., "MICHEL MOORE, an individual"). In this complaint the police chief is sued only in his official capacity, while the five officers are sued both individually and officially.

### `evidence_df`: one row per extracted value

In [ ]:
evidence_df.head()

**Example (selected rows):**

| case_id | field | value | page | quote | status |
|---|---|---|---|---|---|
| 1:22-cv-04065 | case_number | 1:22-cv-04065-MHC | 1 | Civil Action No. 1:22-cv-04065- MHC | verified |
| 1:22-cv-04065 | date_filed | 12/30/22 | 1 | Case 1:22-cv-04065-MHC Document 24 Filed 12/30/22 | verified |
| 1:22-cv-04065 | plaintiffs[0].race | Black | 5 | Mr. André and Mr. English are both Black men | verified |
| 1:22-cv-04065 | plaintiffs[0].age | | | | not stated |
| 1:22-cv-04065 | defendants[1].name | KEVIN ROBERTS | 1 | KEVIN ROBERTS, in his official capacity as Chief of the Clayton County Police Department; | verified |

### `quality_df`: data quality by attribute

In [ ]:
quality_df

Across a library of complaints, this table shows where the pipeline needs attention:

- A high **"failed"** share for an attribute points to a retrieval or prompt problem.
- A high **"not stated"** share shows how often complaints omit that information.

## Saving the Results

All tables are saved to the complaint folder in Google Drive, as CSV files.

In [ ]:
def save_tables(tables, folder, prefix, extracted_df=None):
    xlsx_path = os.path.join(folder, f"{prefix}_analysis.xlsx")
    saved = []
    with pd.ExcelWriter(xlsx_path) as writer:
        for name, df in tables.items():
            keep_index = (name == "quality")
            df.to_excel(writer, sheet_name=name, index=keep_index)
            path = os.path.join(folder, f"{prefix}_{name}.csv")
            df.to_csv(path, index=keep_index, encoding="utf-8-sig")
            saved.append(path)
    if extracted_df is not None:
        path = os.path.join(folder, f"{prefix}_extracted.csv")
        extracted_df.to_csv(path, index=False, encoding="utf-8-sig")
        saved.append(path)
    print("Saved to", folder)
    for path in saved + [xlsx_path]:
        print("  ", os.path.basename(path))

save_tables(tables, COMPLAINT_FOLDER, os.path.splitext(os.path.basename(PDF_PATH))[0], extracted_df=complaint_df)

## Batch Processing

`process_folder` runs the pipeline on every complaint in a folder:

- Results are saved after each file, so an interruption does not lose completed work.
- At the end, all results are combined into the analysis-ready tables and saved as `library_*.csv` and `library_analysis.xlsx`.



*The one-at-a-time `process_folder` is kept for reference; the parallel, resumable version below is the one that runs.*

In [ ]:
def process_folder(folder, output_csv="complaints_extracted.csv", audit_csv="complaints_audit.csv", limit=None):
    pdfs = sorted(glob.glob(os.path.join(folder, "*.pdf")))[:limit]
    all_rows, all_audits, errors = [], [], []

    for i, path in enumerate(pdfs, start=1):
        print(f"\n[{i}/{len(pdfs)}] {os.path.basename(path)}")
        try:
            df, audit = process_complaint(path, verbose=False)
            all_rows.append(df)
            all_audits.append(audit)
        except Exception as e:                     # keep going; record the failure
            errors.append({"source_file": os.path.basename(path), "error": str(e)})
            print("   ERROR:", e)
        if all_rows:                               # save progress after every file
            pd.concat(all_rows, ignore_index=True).to_csv(
                os.path.join(folder, output_csv), index=False, encoding="utf-8-sig")
            pd.concat(all_audits, ignore_index=True).to_csv(
                os.path.join(folder, audit_csv), index=False, encoding="utf-8-sig")

    print(f"\nDone: {len(all_rows)} processed, {len(errors)} errors")
    if not all_rows:
        return {}, pd.DataFrame(errors)

    # Analysis-ready tables for the whole library
    combined_audit = pd.concat(all_audits, ignore_index=True)
    batch_tables = build_analysis_tables(combined_audit)
    save_tables(batch_tables, folder, "library")
    return batch_tables, pd.DataFrame(errors)


# One complaint at a time (slow); the parallel version below is recommended.
# batch_tables, batch_errors = process_folder(COMPLAINT_FOLDER)

### Parallel Batch Processing (recommended)

Most of the time per complaint is spent waiting for API responses, so `process_folder_parallel` runs several complaints at once (`workers`, default 4). Each complaint has its own vector-store collection, so the runs don't interfere, and the extracted values are the same as with `process_folder`; only the order of work changes. Transient API errors (e.g. a 401 from a gateway, rate limits, timeouts) are retried automatically, up to three times per complaint with increasing pauses.

**Speed:** use a **GPU runtime** (Runtime → Change runtime type → T4 GPU): the reranker then runs on the GPU, the slowest step on a CPU. If the API reports rate-limit errors, lower `workers` to 2 or 3.

In [ ]:
# ---- Parallel batch processing ----------------------------------------------------------------------
# Most of the time per complaint is spent waiting for API responses, so several complaints can run at once.
# Each complaint gets its own vector-store collection, so the runs do not interfere; progress files are
# written under a lock. Results are the same as with process_folder (only the order of work changes).
import threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed


def process_folder_parallel(folder, workers=4, output_csv="complaints_extracted.csv",
                            audit_csv="complaints_audit.csv", limit=None, files=None):
    """Like process_folder, but processes `workers` complaints at a time.
    files: optional list of file names to (re)process, e.g. the ones that failed. Their results are then
    merged into the existing library_analysis.xlsx, replacing any earlier rows for the same files."""
    if files is not None:                               # a retry must not overwrite the full run's files
        output_csv, audit_csv = output_csv.replace(".csv", "_retry.csv"), audit_csv.replace(".csv", "_retry.csv")
    pdfs = sorted(glob.glob(os.path.join(folder, "*.pdf")))
    if files is not None:
        pdfs = [p for p in pdfs if os.path.basename(p) in set(files)]
    pdfs = pdfs[:limit]
    results, errors, lock = {}, [], threading.Lock()
    start = time.time()

    def run(path, attempts=3):
        """Process one complaint; transient API errors (401 from a gateway, rate limits, timeouts, 5xx) are
        retried after a pause, since they usually pass."""
        t0 = time.time()
        for attempt in range(1, attempts + 1):
            try:
                df, audit = process_complaint(path, verbose=False)
                return path, df, audit, time.time() - t0
            except Exception as e:
                msg = str(e)
                transient = any(s in msg for s in ("401", "429", "Rate limit", "rate_limit", "timeout", "Timeout",
                                                   "500", "502", "503", "504", "Connection"))
                if not transient or attempt == attempts:
                    raise
                time.sleep(30 * attempt)

    def save_progress():
        done = [results[p] for p in pdfs if p in results]
        if done:
            pd.concat([d for d, _ in done], ignore_index=True).to_csv(
                os.path.join(folder, output_csv), index=False, encoding="utf-8-sig")
            pd.concat([a for _, a in done], ignore_index=True).to_csv(
                os.path.join(folder, audit_csv), index=False, encoding="utf-8-sig")

    print(f"Processing {len(pdfs)} complaints, {workers} at a time")
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(run, p): p for p in pdfs}
        for k, fut in enumerate(as_completed(futures), start=1):
            path = futures[fut]
            name = os.path.basename(path)
            try:
                _, df, audit, secs = fut.result()
                with lock:
                    results[path] = (df, audit)
                    save_progress()
                print(f"[{k}/{len(pdfs)}] done in {secs / 60:.1f} min: {name}")
            except Exception as e:                    # keep going; record the failure
                errors.append({"source_file": name, "error": str(e)})
                print(f"[{k}/{len(pdfs)}] ERROR: {name}: {str(e)[:150]}")

    print(f"\nDone: {len(results)} processed, {len(errors)} errors, in {(time.time() - start) / 60:.1f} min")
    if not results:
        return {}, pd.DataFrame(errors)
    combined_audit = pd.concat([results[p][1] for p in pdfs if p in results], ignore_index=True)
    batch_tables = build_analysis_tables(combined_audit)
    main_path = os.path.join(folder, "library_analysis.xlsx")
    if files is not None and os.path.exists(main_path):
        merge_into_library(main_path, batch_tables)
    else:
        save_tables(batch_tables, folder, "library")
    return batch_tables, pd.DataFrame(errors)


def merge_into_library(main_path, new_tables):
    """Replace the rows of the re-processed complaints in library_analysis.xlsx with the new results."""
    main = pd.read_excel(main_path, sheet_name=None)
    with pd.ExcelWriter(main_path) as w:
        for sheet, df in main.items():
            new = new_tables.get(sheet)
            if new is not None and sheet != "quality" and "source_file" in df.columns and "source_file" in new.columns:
                df = df[~df["source_file"].isin(new["source_file"])]
                df = pd.concat([df, new], ignore_index=True)
            df.to_excel(w, sheet_name=sheet, index=False)
    n = pd.read_excel(main_path, sheet_name="cases")["source_file"].nunique()
    print(f"Merged into {os.path.basename(main_path)}: now {n} complaints")

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), "| extraction model:", EXTRACTION_MODEL, "| RUN:", RUN)
# On a CPU runtime the reranker competes for the same 2 CPU cores, so parallel runs slow each other down:
# 4 workers with a GPU, 2 without
WORKERS = 4 if torch.cuda.is_available() else 2
print("Workers:", WORKERS)

### Start a Folder Over (only when needed)

The run skips complaints already saved in `_results/`. If the set of complaints changed after re-sorting (e.g. wrong documents removed after screening), or the pipeline changed, clear the folder's saved results first. Run with `confirm=False` to see what would be deleted, then with `confirm=True`.

In [ ]:
def reset_folder(folder=None, confirm=False):
    """Delete the saved results (_results/) and the combined tables of a folder, so it is processed from scratch."""
    import shutil
    folder = folder or FOLDERS["full"]
    targets = [os.path.join(folder, "_results")] + [os.path.join(folder, f) for f in os.listdir(folder)
                                                    if f.startswith(("library_", "complaints_extracted"))]
    targets = [t for t in targets if os.path.exists(t)]
    for t in targets:
        print(("deleting " if confirm else "would delete ") + t)
        if confirm:
            shutil.rmtree(t) if os.path.isdir(t) else os.remove(t)
    if not targets:
        print("Nothing to delete in", folder)


# reset_folder(FOLDERS["full"], confirm=False)     # preview
# reset_folder(FOLDERS["full"], confirm=True)      # delete

### Check (or Switch) the Folder Before Processing

Shows which folder the next cell will process. To switch between runs **without re-running the whole notebook** (e.g. if Settings ran with the other value), change `RUN` here and run this cell, then the next one. Expected for the full run: `RUN = full: 466 PDFs in …/COMPLAINTS_SORTED/complaints`.

In [ ]:
RUN = "full"                       # "full" or "validation"
COMPLAINT_FOLDER = FOLDERS[RUN]
PDFS = sorted(glob.glob(os.path.join(COMPLAINT_FOLDER, "*.pdf")))
print(f"RUN = {RUN}: {len(PDFS)} PDFs in {COMPLAINT_FOLDER}")

### OCR for Scanned Complaints (image-only pages)

Some complaints are scans: the court's electronic filing system stamps a **text header** on each page ("Case … Document … Filed … Page x of y"), but the body is an image. The pages then *look* like text to a simple check, while the pipeline can read nothing but the header (the plaintiff's name, gender and race come out empty).

This cell finds such files (fewer than 200 characters per page once the header is removed), adds a text layer with OCR (`ocrmypdf --force-ocr`; the original scans are kept in `COMPLAINTS_SORTED/scanned_originals/`), and removes the saved results of the files it OCR'd, so the next cell re-extracts **only those**. It is safe to run again: an OCR'd file has a text layer and is not selected a second time. About a minute per scanned complaint.

*Note: re-running Step 3 of the sorting notebook re-copies the original downloads into `complaints/` (and clears this folder), which would undo the OCR.

In [ ]:
import subprocess, shutil, sys, importlib.util
import fitz
from tqdm.auto import tqdm

HEADER_RE = re.compile(r"Case\s+\S+\s+Document\s+\S+.*?Page\s+\d+\s+of\s+\d+", re.I | re.S)


def body_chars(path, pages=5):
    """Average characters per page on the first pages, after removing the court's header stamp."""
    with fitz.open(path) as d:
        n = min(pages, len(d))
        return sum(len(HEADER_RE.sub("", d[i].get_text()).strip()) for i in range(n)) / max(n, 1)


scanned = [p for p in sorted(glob.glob(os.path.join(COMPLAINT_FOLDER, "*.pdf"))) if body_chars(p) < 200]
print(f"{len(scanned)} scanned complaint(s) without a readable text layer")
if scanned:
    # OCR tools: Tesseract and Ghostscript from apt, OCRmyPDF from pip into this notebook's Python
    # (the apt version of OCRmyPDF uses the system Python, whose pikepdf fails to load in Colab)
    if shutil.which("tesseract") is None or shutil.which("gs") is None:
        subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr ghostscript > /dev/null",
                       shell=True, check=True)
    if importlib.util.find_spec("ocrmypdf") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ocrmypdf"], check=True)
    backup = os.path.join(os.path.dirname(COMPLAINT_FOLDER.rstrip("/")), "scanned_originals")
    os.makedirs(backup, exist_ok=True)
    done = []
    for p in tqdm(scanned, desc="OCR", unit="pdf"):
        name = os.path.basename(p)
        if not os.path.exists(os.path.join(backup, name)):
            shutil.copy2(p, os.path.join(backup, name))           # keep the original scan
        out = p + ".ocr.pdf"
        r = subprocess.run([sys.executable, "-m", "ocrmypdf", "--force-ocr", "-l", "eng", "--quiet", p, out], capture_output=True, text=True)
        if r.returncode == 0:
            os.replace(out, p)
            done.append(p)
        else:
            print("OCR failed:", name, r.stderr[-200:])
    removed = 0
    for p in done:                                                # re-extract the OCR'd complaints
        res = os.path.join(COMPLAINT_FOLDER, "_results", os.path.splitext(os.path.basename(p))[0] + ".pkl")
        if os.path.exists(res):
            os.remove(res)
            removed += 1
    print(f"OCR done: {len(done)} | saved results removed: {removed} -> the next cell re-extracts them")
    for p in done:
        print("  ", os.path.basename(p))

### Processing the Folder (both runs): saved complaint by complaint, resumable

**Yes, a long run needs this.** Colab sessions end after a few hours (or when idle), so 466 complaints can't be relied on to finish in one session. This cell handles it:

- **Each finished complaint is saved immediately** to `_results/` in the complaint folder (one file per complaint, written safely, so a stopped session never leaves a half-saved result).
- **Resuming:** if the session stops, open the notebook again, run it to this cell, and run this cell: complaints already saved are skipped, and the run continues with the rest. Nothing finished is processed twice.
- **The combined tables** (`library_analysis.xlsx`, `library_*.csv`, `complaints_extracted.csv`) are rebuilt from all saved results **after every batch** (`BATCH_SIZE`, default 25) and at the end, so they are always up to date within one batch. `rebuild_library()` can also be run on its own at any time.
- **Failures:** a complaint that fails (after automatic retries of transient API errors) is listed at the end; run the cell again to retry it.
- **Progress:** each complaint prints its time and an estimate of the time left.

To start a folder over from scratch, delete its `_results/` subfolder and `library_analysis.xlsx`.

**Test run first:** with `TEST_LIMIT = 5` (the default in the cell), only 5 complaints are processed, which shows the time per complaint and, on your OpenAI Usage page, the cost. They are saved, so the full run skips them. Then set `TEST_LIMIT = None` and run the cell again.

**Pro se flag:** the combined `cases` table gets a column `pro_se` (True when the complaint's first or last pages say "pro se", "in pro per" or similar). Self-written complaints are often in the first person and state no gender, race or age, so this helps interpret "not stated" values. It is a text check, not a model call, so it doesn't change the validated extraction.

In [ ]:
# ---- Processing the library: saved complaint by complaint, resumable ----------------------------------
RESULTS_DIR = os.path.join(COMPLAINT_FOLDER, "_results")   # one file per finished complaint (the record of progress)
BATCH_SIZE = 25                                            # the combined tables are rebuilt after each batch


def _result_path(name):
    return os.path.join(RESULTS_DIR, os.path.splitext(name)[0] + ".pkl")


def finished_files():
    """Complaints already processed: a result file in _results/. Only for a folder processed before results were
    saved one by one (no _results/ yet) do the rows of library_analysis.xlsx count as processed."""
    if os.path.isdir(RESULTS_DIR) and any(f.endswith(".pkl") for f in os.listdir(RESULTS_DIR)):
        return {f[:-4] for f in os.listdir(RESULTS_DIR) if f.endswith(".pkl")}
    lib = os.path.join(COMPLAINT_FOLDER, "library_analysis.xlsx")
    if os.path.exists(lib):
        return {os.path.splitext(f)[0] for f in pd.read_excel(lib, sheet_name="cases")["source_file"]}
    return set()


def _process_and_save(path, attempts=3):
    """Process one complaint (retrying transient API errors) and save its result at once, so a finished
    complaint is never lost if the session stops. The file is written under a temporary name and then renamed,
    so an interrupted write never leaves a half-saved result."""
    t0 = time.time()
    for attempt in range(1, attempts + 1):
        try:
            df, audit = process_complaint(path, verbose=False)
            break
        except Exception as e:
            msg = str(e)
            transient = any(s in msg for s in ("401", "429", "Rate limit", "rate_limit", "timeout", "Timeout",
                                               "500", "502", "503", "504", "Connection"))
            if not transient or attempt == attempts:
                raise
            time.sleep(30 * attempt)
    target = _result_path(os.path.basename(path))
    pd.to_pickle({"complaint": df, "audit": audit}, target + ".tmp")
    os.replace(target + ".tmp", target)
    return time.time() - t0


PRO_SE_RE = re.compile(r"\bpro(?:\s+|\s*-\s*)se\b|in\s+pro\s+per\b|in\s+propria\s+persona|self[- ]represented", re.I)


def detect_pro_se(path):
    """True if the complaint's first or last two pages identify the plaintiff as self-represented ("pro se",
    "in pro per", "in propria persona"); a deterministic text check, not a model call."""
    try:
        import fitz
        with fitz.open(path) as d:
            pages = sorted(set(range(min(2, len(d)))) | set(range(max(0, len(d) - 2), len(d))))
            text = " ".join(d[i].get_text() for i in pages)
    except Exception:
        return None
    return bool(PRO_SE_RE.search(text))


def rebuild_library():
    """Combine all saved results into the analysis tables (library_analysis.xlsx and the library_*.csv files).
    Rows from an earlier library_analysis.xlsx for complaints without a result file are kept."""
    files = sorted(glob.glob(os.path.join(RESULTS_DIR, "*.pkl")))
    if not files:
        return 0
    saved = [pd.read_pickle(f) for f in files]
    tables = build_analysis_tables(pd.concat([s["audit"] for s in saved], ignore_index=True))
    # pro se flag: first-person, self-written complaints often state no gender, race or age
    tables["cases"]["pro_se"] = tables["cases"]["source_file"].map(
        lambda f: detect_pro_se(os.path.join(COMPLAINT_FOLDER, f)))
    pd.concat([s["complaint"] for s in saved], ignore_index=True).to_csv(
        os.path.join(COMPLAINT_FOLDER, "complaints_extracted.csv"), index=False, encoding="utf-8-sig")
    lib = os.path.join(COMPLAINT_FOLDER, "library_analysis.xlsx")
    if os.path.exists(lib) and len(set(pd.read_excel(lib, sheet_name="cases")["source_file"])
                                   - set(tables["cases"]["source_file"])):
        merge_into_library(lib, tables)              # keep complaints from earlier runs
    else:
        save_tables(tables, COMPLAINT_FOLDER, "library")
    return pd.read_excel(lib, sheet_name="cases")["source_file"].nunique()


def process_library(folder=None, workers=None, batch_size=None, limit=None):
    """Process every complaint in the folder that has not been processed yet, `workers` at a time. Each finished
    complaint is saved immediately; the combined tables are rebuilt after each batch and at the end.
    If the session stops, run this cell again: it continues with the remaining complaints.
    limit: process at most this many of the remaining complaints (e.g. 5 for a test run)."""
    folder = folder or COMPLAINT_FOLDER
    workers = workers or WORKERS
    batch_size = batch_size or BATCH_SIZE
    os.makedirs(RESULTS_DIR, exist_ok=True)
    done = finished_files()
    pdfs = sorted(glob.glob(os.path.join(folder, "*.pdf")))
    todo = [p for p in pdfs if os.path.splitext(os.path.basename(p))[0] not in done]
    remaining = len(todo)
    if limit:
        todo = todo[:limit]
    print(f"Folder: {folder}")
    print(f"{len(pdfs)} complaints: {len(pdfs) - remaining} already processed, {remaining} to do"
          + (f"; this run: {len(todo)} (limit)" if limit else "") + f", {workers} at a time")
    failed, start, finished_now = [], time.time(), 0
    for b in range(0, len(todo), batch_size):
        batch = todo[b:b + batch_size]
        print(f"\n=== Batch {b // batch_size + 1} of {-(-len(todo) // batch_size)} ({len(batch)} complaints) ===")
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(_process_and_save, p): p for p in batch}
            for fut in as_completed(futures):
                name = os.path.basename(futures[fut])
                try:
                    secs = fut.result()
                    finished_now += 1
                    rate = (time.time() - start) / finished_now
                    left = (len(todo) - finished_now - len(failed)) * rate / 60
                    print(f"  done ({finished_now}/{len(todo)}) in {secs / 60:.1f} min: {name}"
                          f"  | about {left:.0f} min left")
                except Exception as e:
                    failed.append(name)
                    print(f"  ERROR: {name}: {str(e)[:150]}")
        n = rebuild_library()
        print(f"Tables rebuilt: library_analysis.xlsx has {n} complaints")
    print(f"\nFinished this run: {finished_now} processed, {len(failed)} failed, "
          f"{(time.time() - start) / 60:.1f} min. Failed: {failed or 'none'}")
    if failed:
        print("Run this cell again to retry the failed complaints.")
    return failed


# Test first: TEST_LIMIT = 5 processes 5 complaints and stops (they are saved, so the full run skips them).
# Then set TEST_LIMIT = None and run this cell again for all remaining complaints.
TEST_LIMIT = 5

failed = process_library(COMPLAINT_FOLDER, limit=TEST_LIMIT)
if TEST_LIMIT:
    done_now = [f for f in os.listdir(RESULTS_DIR) if f.endswith(".pkl")]
    print(f"\nTest run finished ({len(done_now)} complaints saved in total). Check the times above and the"
          " Usage page of your OpenAI account; estimate: average minutes per complaint x remaining / workers."
          " For the full run, set TEST_LIMIT = None and run this cell again.")
lib_path = os.path.join(COMPLAINT_FOLDER, "library_analysis.xlsx")
if os.path.exists(lib_path):
    display(pd.read_excel(lib_path, sheet_name="plaintiffs")["race_std"].value_counts())

## Limitations and Next Steps

- **Verified is not the same as correct.** The citation check confirms that a value is supported by a quote, not that the right passage was chosen. In a complaint describing several encounters, the pipeline coded a verified street-corner passage, while the hand coding placed the encounter at issue in a parked car. Accuracy is therefore measured against a hand-coded gold set (`03_Validation.ipynb`).
- **Absent values are not verified.** A value wrongly reported as "not stated" (retrieval missed the passage) is not caught; `citation_summary` lists these for spot-checking. Facts stated far from the parties section are hardest, for the pipeline and for human coders alike.
- **Strict subject check.** Gender or age stated only through a pronoun is accepted when the pronoun is within about five words of the plaintiff's name or "Plaintiff"; this prevents misattribution (e.g., "a female officer") but misses pronouns elsewhere in a paragraph, which human coders read at the paragraph level.
- **Plaintiffs are not always the people in the encounter.** In wrongful-death and representative cases the plaintiffs are relatives or estates, and organizations can be plaintiffs; victim-level fields would be needed to analyze who was profiled.
- **Group descriptions are applied to each plaintiff,** which can misassign values when a group is mixed (e.g., "Black and Latino residents").
- **Long files are indexed up to page 60** (`MAX_PAGES_INDEXED`): this covers the parties and facts in nearly all complaints, but values stated only later in very long complaints are missed.
- **Document selection affects extraction.** The pipeline reads the file the sorting stage selected; in the validation sample, 2 of 21 selected files were not the complaint.
- **"Date filed" is the document's filing date,** not the case's (for amended complaints, the case opened earlier). Case numbers are kept as printed.
- **Rule-based coding** of race categories and defendant types (Entity / Individual / Doe) uses keyword patterns; review "Unmapped" race values and a sample of defendant types. Defendant lists were not validated independently (the gold-set defendant lists were prepared with AI assistance).
- **Scanned and handwritten complaints:** OCR'd text is matched more tolerantly, slightly weakening the citation check; handwriting is read poorly and was excluded from the validation set.
- **The setting codebook (v0.3) has been applied by one coder** on 21 complaints; inter-coder reliability has not been measured. Complaints with several incidents are summarized at the complaint level.
- **Model and runtime.** Results apply to the validated configuration (gpt-4o, text-embedding-3-small, bge-reranker-base); changing the model requires validating again. On a CPU, a complaint takes about 10 minutes; the full library needs a GPU, and routing through an API gateway caused intermittent authentication errors, handled by automatic retries.
- **Validation is a pilot** on 19 complaints [key-variable accuracy: to be added from `03_Validation.ipynb`], so confidence intervals are wide. **Next steps:** adjudicate disagreements, add a second coder, compare models (e.g., gpt-4o vs. gpt-4o-mini) on the gold set, validate an LLM judge, and run the full library of 466 complaints.